In [1]:
import pandas as pd
import numpy as np
import os
import time
import pickle
import itertools
import seaborn as sns
from itertools import combinations,chain
import umap
from matplotlib import pyplot as plt

#%run s7_0_functions_for_data_preprocessing_for_ml.ipynb
os.environ["NUMBA_WARNINGS"] = "1"

os.chdir(os.getcwd())
os.getcwd()
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action="ignore", category=pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action="ignore", category=DeprecationWarning)

from itables import init_notebook_mode
init_notebook_mode(connected=True,all_interactive=True)

# Functions and parameters

In [2]:
parameters_for_analysis={'tb21_22_2984_pats_22_vars_result_at_end_of_treatment':{
                            'result_cat':'RESULT_AT_END_OF_TREATMENT',
                            'fn':'tb21_22_2984_pats_22_vars_result_at_end_of_treatment'},

                         'tb21_22_2984_pats_22_vars_relapse_without_dr_reg':{
                            'result_cat':'RELAPSE',
                            'fn':'tb21_22_2984_pats_22_vars_result_at_end_of_treatment'},
    
                       
                         
                         'tb21_1405_pats_40_vars_result_at_end_of_treatment':{
                            'result_cat':'RESULT_AT_END_OF_TREATMENT',
                            'selection_method':'patient_clustering',
                            'clust_comb':'1-3-4-5',
                            'graph_metric':None,
                            'num_of_common_vars':40,
                            'training_days':120}}


def load_merged_data_of_lab_vars():
    #load patient IDs who are considered in this  analysis
    pat_id_df=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)
    # get all pat ids
    all_ids=pat_id_df['USUBJID'].to_list()

    fname='merged_df.csv.gz'
    
    fn=os.path.join('../data/',fname)
    merged_df=pd.read_csv(fn,low_memory=False,index_col=0)

    return merged_df


#### -----------------------------------
def subset_columns_relevant_for_analysis(data,pats_for_analysis,common_variables_for_analysis,return_selected_columns_only):
    # Read lab test variable type information (numerical or categorical)
    with open('../data/lab_variables.pkl', 'rb') as f:
        lab_variable_type_dict = pickle.load(f)
    
    ## Correct typos and drop some duplicated columns
    #data.columns = ['WEEK' if x=='Week' else x for x in data.columns]
    
    ### SELECT COLUMNS THAT HOLD THE RESULT INFROMATION USED IN THE ANALYSIS FOR EACH DATASET TYPE
    dataset_types_of_common_vars=list(set([x.split('_')[0] for x in common_variables_for_analysis if '_' in x]))

    result_column_names_per_dataset_type={  
                
                'mb':{'categorical':{'categorical_vars':['Identification','Culture Growth','Categorical Count','Unknown',\
                                                        'Colony Count, Categorical','MPT64 Antigen Test',
                                                         
                                                        # Standardised variable names 
                                                         'ZN-smear','MGIT','HAIN-test','MTB-complex',
                                                         'Auramine-smear','LJ-culture','AccuProbe',
                                                         'MPT64-Antigen-Test','RT-PCR'],
                                     
                                    'categorical_result_suffix':['RESULT','CULTURE_STATUS','STD_CAT_ORDINAL_RESULT']},
                      
                    'numerical':{'numerical_vars':['Time to Detection','Colony Count','Minimum Cycle Threshold of Detection','MGIT','LJ-culture'],
                                'numerical_result_suffix':['STD_NUM_RESULT','STD_NUM_UNIT']}},

                'pc':{'numerical':{'numerical_result_suffix':['STD_NUM_RESULT','STD_UNITS']}},

                're':{'categorical':{'categorical_vars':['Cavitation','X-Ray compatible with TB','Development of New Lesions',\
                                                        'Extension of Old Lesions','Extent of Disease','Interpretation',\
                                                        'Size Of Cavitation','Detailed Classification'],
                                    'categorical_result_suffix':['STD_CAT_RESULT']},
                    #'numerical':{'numerical_vars':['Detailed Classification'],
                    #            'numerical_result_suffix':['STD_NUM_RESULT']}
                                                                             },
        
                'vs':{'numerical':{'numerical_result_suffix':['STD_NUM_RESULT','STD_NUM_UNIT']}},

                'lb':{'categorical':{'categorical_vars':lab_variable_type_dict['lb_categorical_vars'],
                                    'categorical_result_suffix':['STD_CAT_ORDINAL_RESULT']},
                    'numerical':{'numerical_vars':lab_variable_type_dict['lb_numerical_vars'],
                                #'numerical_result_suffix':['STD_NUM_RESULT_SCALED']},
                                'numerical_result_suffix':['STD_NUM_RESULT','STD_NUM_UNIT']},
                                 },

                #'mh':{'categorical':{'categorical_result_suffix':}},

                'ce':{'categorical':{'categorical_result_suffix':['STD_CAT_RESULT','STD_CETOXGR']}},

                'mr':{'categorical':{'categorical_result_suffix':['STD_CAT_RESULT']}},

                'mic':{'numerical':{'numerical_result_suffix':['STD_NUM_RESULT','MSSTRESU']}},

                'ms':{'categorical':{'categorical_result_suffix':['STD_CAT_RESULT']},
                    'numerical':{'numerical_result_suffix':['SUSC_CONC','RESISTANCE_CONC']}},

                'su':{'categorical':{'categorical_result_suffix':['STD_CAT_RESULT']}}}

    ### Collect the columns to keep for analysis from the dataframe containing the previously selected variables
    columns_for_analysis={}

    ## Loop over the common variables among patients selected in previous step, and using the dictionary above
    #  extract the columns that hold usable result information for the analysis
    datasets_to_loop_over=set([*result_column_names_per_dataset_type])&set(dataset_types_of_common_vars)

    for dataset_type in datasets_to_loop_over:
        columns_for_analysis[dataset_type]=[]
        dataset_type_vars_all=[x for x in common_variables_for_analysis if x.startswith(dataset_type+'_')]
        
        for dataset_type_var in dataset_type_vars_all:
            stripped_dataset_type_var=dataset_type_var.split('_')[1]

            for var_type in result_column_names_per_dataset_type[dataset_type].keys():

                if var_type+'_vars' in result_column_names_per_dataset_type[dataset_type][var_type].keys():

                    if stripped_dataset_type_var in result_column_names_per_dataset_type[dataset_type][var_type][var_type+'_vars']:
                        for suffix in result_column_names_per_dataset_type[dataset_type][var_type][var_type+'_result_suffix']:
                            columns_for_analysis[dataset_type].append('_'.join([dataset_type_var,suffix]))

                elif var_type+'_vars' not in result_column_names_per_dataset_type[dataset_type][var_type].keys(): 
                    for suffix in result_column_names_per_dataset_type[dataset_type][var_type][var_type+'_result_suffix']:
                        columns_for_analysis[dataset_type].append('_'.join([dataset_type_var,suffix]))


    ## Extract column holding drug regimen cumulative dose information
    temporal_cols=data.columns[data.columns.str.startswith(('ae_','cmdos_','cmday_','cmind_','mh_','dr_reg_'))].tolist()

    ## Create one list of the columns_for_analysis dictionary values containing variables from the different dataset types
    columns_for_analysis_list=list(chain(*np.array(list(columns_for_analysis.values()),dtype=object)))

    ## Columns names of patients extracted from dm dataframe
    dm_colnames=['USUBJID','DAY','STUDYID','ARM','AGE','SEX','RACE']
    cols_for_analysis=dm_colnames+ columns_for_analysis_list + temporal_cols


    ## If return_columns_only -> select and return COLUMNS ONLY for analysis from the provided dataset
    if return_selected_columns_only==True:

        data_for_anal=data.loc[:,data.columns.isin(cols_for_analysis)]

        ## Drop mb columns, that are culture based and don't give an instant result,
        #  or the method is unknown -> KEEP MGIT MEASUREMENTS, AS THEY ARE THE GOAL OF PREDICTION
        #mb_cols_to_drop='|'.join(['mb_Culture Growth','mb_Categorical Count','mb_Colony Count','mb_Colony Count, Categorical','mb_Unknown'])
        cols_for_analysis=data_for_anal.columns
                
        return cols_for_analysis

    
    ## If return_columns_only is False -> select columns for analysis and return the DATASET with selected columns
    if return_selected_columns_only==False:
        # Drop all NaN columns and duplicates
        data_for_anal=data.loc[data['USUBJID'].isin(pats_for_analysis),cols_for_analysis].dropna(how='all',axis=1)

        ## Drop mb columns, that are culture based and don't give an instant result,
        #  or the method is unknown -> KEEP MGIT MEASUREMENTS, AS THEY ARE THE GOAL OF PREDICTION
        #mb_cols_to_drop='|'.join(['mb_Culture Growth','mb_Categorical Count','mb_Colony Count','mb_Colony Count, Categorical','mb_Unknown'])
        #data_for_anal=data_for_anal.loc[:,~data_for_anal.columns.str.contains(mb_cols_to_drop,na=False)]
        
        
        return data_for_anal





##=========================================  
## For patients, who stopped therapy earlier than the scheduled duration of the study, the cumulative drug doses are set to 0 for those days, 
#. where the drugs weren't taken anymore. This originates from the way the drug regimen was extracted in step s4. 
#  To remedy this problem, forward fill the last cumulative dose for those days.
def ffill_dr_reg_cumul_cols(dr_reg):
    ## Extract dr_reg cumulative columns + DAY and USUBJID
    dr_reg_ffill_cols=['DAY','USUBJID']+dr_reg.columns[dr_reg.columns.str.contains('cumul')].tolist()

    ## 1. Replace the 0s with NaNs==> first therapy day & days where wasn't taken anymore are becoming NaNs
    ## 2. Forward fill ==> only the days without drug threapy get filled with last cumulative dose
    ## 3. Fill NaNs with 0==> fill the first day of therapy with a 0, indicating no drugs were taken yet
    dr_reg_ffill=dr_reg.loc[:,dr_reg_ffill_cols].groupby('USUBJID',as_index=False).apply(lambda x: x.sort_values(by='DAY').replace(0, np.nan).ffill().fillna(0))

    ## Merge the original data with the ffilled drug regimen data
    dr_reg_ffill_=pd.merge(dr_reg.loc[:,~dr_reg.columns.str.contains('cumul')],\
                            dr_reg_ffill.loc[:,dr_reg_ffill_cols],on=['DAY','USUBJID'],how='outer')

    return dr_reg_ffill_

### -------------------------------------------------------
##  Add previously created temporal dataframes (__ae,cm,dr_reg__) + 
#   static medical history (__mh__) to __merged_df__ (dataframe containing previously merged laboratory variables)

def concatenate_temporal_data(pats_for_analysis,keep_data_with_unknown_drug_regimen,
                             common_variables_for_analysis,
                             keep_days_with_lab_measurements_only):
    from itertools import chain
    start = time.time()
    ## Load whole dataset
    all_phase_df=load_merged_data_of_lab_vars()
    
    ## Load the last day of therapy of initial treatment
    last_initial_therapy_day_df=pd.read_csv('../data/out_last_initial_therapy_day_list_1018_20_21_22_30.csv.gz',index_col=0)
    
    ## Subset merged dataframe to patients who are considered for analysis
    df_for_anal=all_phase_df.loc[all_phase_df['USUBJID'].isin(pats_for_analysis),:]#.dropna(how='all',axis=1)
    end=time.time()
    t=round((end-start)/60,2)
    print('lab measurement data: done ',t, ' minutes')  
    print('dataframe shape: ',df_for_anal.shape)  

    
    ### LOOP OVER TEMPORAL DATASETS AND ADD THEM TO THE DATAFRAME CREATED IN PREVIOUS STEP

    temp_df_fnames={'dr_reg':'out_temporal_pat_regimens_1018_20_21_22_30.csv.gz',
                    'ae_temp':'out_ae_standardised_temporal.csv.gz',
                    'cm_temp_drugs_doses':'out_cm_temporal_with_doses.csv.gz',
                    'cm_temp_drugs_days_of_appl':'out_cm_temporal_days_of_application.csv.gz',
                    'cm_temp_ind':'out_cm_temporal_indications.csv.gz'}                    

    
    for temp_df_name in [*temp_df_fnames][0:]:

        fn=os.path.join('../data',temp_df_fnames[temp_df_name])
        if temp_df_name=='dr_reg':
            temp_df=pd.read_csv(fn,low_memory=False,index_col=0)
            
            ## Forward fill cumulative columns
            temp_df=ffill_dr_reg_cumul_cols(temp_df)
            #print('dr_reg shape',temp_df.shape)

        ## For these dataframes the pre-selected variables can reduce the numbers of columns that need to be read
        #  Faster read-in time
        if temp_df_name=='ae_temp' or temp_df_name=='cm_temp_ind':
            data_for_cols=pd.read_csv(fn, index_col=0, nrows=0)
            cols_to_read=['DAY','USUBJID']+ list(set(common_variables_for_analysis)&set(data_for_cols.columns))
            temp_df=pd.read_csv(fn,low_memory=False,usecols=cols_to_read,index_col=0)
        
        ## The columns names of these are not in the pre-selected variables, as they are standardised and 
        #  derived in 3_4 -> read all of their columns and drop the unnecessary ones
        if temp_df_name=='cm_temp_drugs_doses' or temp_df_name=='cm_temp_drugs_days_of_appl':  
            temp_df=pd.read_csv(fn,low_memory=False,index_col=0)  


        common_idx=list(set(temp_df['USUBJID'])&set(df_for_anal['USUBJID']))
        temp_df=temp_df.loc[temp_df['USUBJID'].isin(common_idx)].dropna(how='all',axis=1)
        
        if temp_df_name=='dr_reg':
            cols_to_keep=temp_df.columns[temp_df.columns.str.contains('cumul',na=False)].tolist() + ['USUBJID','DAY','STUDYID']
            temp_df=temp_df.loc[temp_df['USUBJID'].isin(pats_for_analysis),cols_to_keep].fillna(0)
            dr_reg_max_days=temp_df.loc[:,['USUBJID','DAY']]
            
            ## KEEP ONLY THE DAYS THAT ARE PRESENT IN df_for_anal -> MERGE DATA ONLY FROM THOSE DAYS in dr_reg
            # i.e. Pat 1 has measurements from day 1, 14, 28, 56, 120, 155 in df_for_anal, whereas in dr_reg 
            #      Pat 1 has consecutive data from day 1-day 182 -> 
            #      to merge data from only the days present in df_for_anal, use 'left' as merging method
            if keep_days_with_lab_measurements_only==True:
                merge_method='left'
            
            # MERGE ON THE DAYS THAT ARE PRESENT IN dr_reg -> 
            # i.e. Pat 1 has measurements from day 1, 14, 28, 56, 120, 155 in df_for_anal, whereas in dr_reg 
            #      Pat 1 has consecutive data from day 1-day 182 -> merge an all days from day 1-182 ->
            #      for this use 'outer' as merging method
            if keep_days_with_lab_measurements_only==False:  
                merge_method='outer'

            df_for_anal=pd.merge(df_for_anal,temp_df,left_on=['USUBJID','DAY','STUDYID'],\
                                right_on=['USUBJID','DAY','STUDYID'],how=merge_method)                             

            if keep_data_with_unknown_drug_regimen==False:
                ## KEEP THERAPY RANGE ONLY, WHERE THERE IS RELIABLE DRUG REGIMEN DATA & MICROBIOLOGICAL MEASUREMENTS AVAILABLE
                comm_idx=list(set(dr_reg_max_days['USUBJID'])&set(df_for_anal['USUBJID']))
                days_to_drop=[]
                for pat in comm_idx[0:]:
                    if pat in last_initial_therapy_day_df['USUBJID'].unique():
                        dr_reg_max=last_initial_therapy_day_df.loc[last_initial_therapy_day_df['USUBJID']==pat,'last_init_therapy_day'].values[0]+10
                    else:
                        dr_reg_max=(dr_reg_max_days[dr_reg_max_days['USUBJID']==pat]['DAY'].max())
                    #mb_max=(df_for_anal[(df_for_anal['USUBJID']==pat)&
                    #                ~(df_for_anal.loc[:,df_for_anal.columns.str.startswith('mb_')].isna().all(axis=1))]['DAY'].max())
                    max_day_to_keep=dr_reg_max #max(mb_max,dr_reg_max) # 
                    days_to_drop.append(df_for_anal[(df_for_anal['USUBJID']==pat)&(df_for_anal['DAY']>max_day_to_keep)].index.tolist())
                days_to_drop=list(chain(*days_to_drop))
                df_for_anal=df_for_anal.drop(index=days_to_drop)                                  
            

        if temp_df_name!='dr_reg':                
            df_for_anal=pd.merge(df_for_anal,temp_df,left_on=['USUBJID','DAY','STUDYID'],\
                                    right_on=['USUBJID','DAY','STUDYID'],how='left')
        
        del temp_df
        end=time.time()
        t=round((end-start)/60,2)
        print(temp_df_name+' done ',t, ' minutes')  
        print('dataframe shape: ',df_for_anal.shape) 
             

    
    ###--------------------
    #### LOAD EXPANDED MEDICAL HISTORY DATAFRAME AND ADD COLUMNS FOR EACH PATIENT 
    mh=pd.read_csv('../data/out_mh_standardised_expanded.csv.gz',low_memory=False,index_col=0)
    ## Drop terms with less than 10 patients having that term in medical history
    mh=mh.loc[:,~mh.columns.str.contains('STUDYID')]
    #mh=mh.loc[:,mh.sum()>10]
    
    common_idx=list(set(mh.index)&set(df_for_anal['USUBJID']))
    mh=mh.loc[common_idx,:].dropna(how='all',axis=1)

    mh_colnames_with_prefix=['mh_'+x for x in mh.columns.tolist()]
    mh.columns=mh_colnames_with_prefix
    common_cols=list(set(mh.columns)&set(common_variables_for_analysis))
    mh=mh.loc[:,common_cols]
    
    ## Add common columns with Nans
    df_for_anal[common_cols]=np.nan
    
    ## Select rows of patients in df_for_anal with MH data
    ids_with_mh=df_for_anal.loc[df_for_anal['USUBJID'].isin(common_idx),'USUBJID'].index.tolist()
    df_for_anal.loc[ids_with_mh,common_cols]=mh.loc[df_for_anal.loc[ids_with_mh,'USUBJID'].tolist(),common_cols].values
    del mh        
    #df_for_anal=df_for_anal.drop(columns=['mh_STUDYID'])

    end=time.time()
    t=round((end-start)/60,2)
    print('mh done ',t, ' minutes')
    print('dataframe shape: ',df_for_anal.shape) 
    
    
    ### Fill NaN datapoints with 0 where no imputation is needed 
    #  (cmdos and cmday columns need forward fill as they are cumulative doses/days of drug application)
    #df_for_anal=df_for_anal.sparse.to_dense()
    #df_for_anal.loc[:,df_for_anal.columns.str.startswith(('ae_','cmind_','mh_'))]=df_for_anal.loc[:,df_for_anal.columns.str.startswith(('ae_','cmind_','mh_'))].fillna(0)

    #end=time.time()
    #t=round((end-start)/60,2)
    #print('fillna with 0s done ',t, ' minutes') 
       
    return df_for_anal

### -------------------------------------------------------
## Return all the variable names in the formats:
#  1. As a dict, where each key corresponds to one dataset type and the values are all the variables for that datset type
#      i.e. {'lb':['lb_Blood Hemoglobin','lb_Blood Urate',...], 'mb':['mb_Time to Detection',...],...}
#  2. As a list containing all the variable names
#     i.e. ['lb_Blood Hemoglobin','lb_Blood Urate',...,'mb_Time to Detection'...,]
def get_all_var_names():
    f = open('../data/all_pat_variables_with_reliable_therapy_data_dict',"rb")
    d=pickle.load(f)

    vars_per_dataset={}
    
    ## Add dm colnames
    #vars_per_dataset['dm']=['AGE','SEX','RACE','ARM']
    all_vars=set()
    for ds_type in [*d]:
        l=[]
        for pat in d[ds_type]:
            l.append(d[ds_type][pat])    
        
        unique_elements = set()
    
        # Loop through each sub-list in the ragged list
        for sublist in l:
            # Add each element to the set
            unique_elements.update(sublist)

        ## Update the dict subsetted per dataset type
        vars_per_dataset[ds_type]=unique_elements
        
        ## Update set containing all variable names
        all_vars.update(unique_elements)
        #print(unique_elements)    
    
    return vars_per_dataset,list(all_vars)



#### =======================
### LOAD COMMON VARIABLES AND PATIENTS FOR THE GIVEN OUTPUT LABEL CATEGORY, CREATED IN S7_2_....IPNYB NOTEBOOK
def return_common_vars_pats_for_anal(result_cat,selection_method,num_of_common_vars,clust_comb,graph_metric): 

    ## Select the common variables and common patients
    if 'clustering' in selection_method:
        if selection_method=='patient_clustering':
            fname='pat_clust_common_vars.pickle'
        if selection_method=='var_clustering':
            fname='var_clust_common_vars.pickle'
        
        ## Use patients graphs to extract common patients and variables
        fn=os.path.join('../data/',fname)
        with open(fn, 'rb') as f:
            common_vars=pickle.load(f)
        
        common_variables_for_analysis=common_vars[result_cat][clust_comb][num_of_common_vars]['common_variables']+['USUBJID','DAY']
        pats_for_analysis=common_vars[result_cat][clust_comb][num_of_common_vars]['patients']
    
    if 'graph' in selection_method:
        if selection_method=='patient_graph':
            fname='pat_gr_common_dict.pickle'
        
        if selection_method=='variable_graph':
            fname='var_gr_common_dict.pickle'  

        ## Use patients graphs to extract common patients and variables
        fn=os.path.join('../data/',fname)
        with open(fn, 'rb') as f:
            gr_common_dict=pickle.load(f)
        num_of_pats=[*gr_common_dict[result_cat][clust_comb][graph_metric][num_of_common_vars]][0]
        common_variables_for_analysis=gr_common_dict[clust_comb][graph_metric][num_of_common_vars][num_of_pats]['common_vars']+['USUBJID','DAY']
        common_variables_for_analysis=list(set(common_variables_for_analysis))
        pats_for_analysis=gr_common_dict[clust_comb][graph_metric][num_of_common_vars][num_of_pats]['common_pats']
  
    return common_variables_for_analysis,pats_for_analysis

#### =======================
def select_temporal_cols_with_suff_pat_data(temp_data_name,X_subset,temp_col_threshold): 
    variables_per_patient_all=pd.read_csv('../data/all_pat_variables_with_reliable_therapy_data.csv.gz',index_col=0)
    
    vars_per_pat=variables_per_patient_all.loc[X_subset['USUBJID'].unique(),:]
    a=vars_per_pat.loc[:,vars_per_pat.columns.str.startswith(temp_data_name)].sum()
    temp_cols_with_suff_data=a[a>len(X_subset['USUBJID'].unique())*temp_col_threshold].index.tolist()

    return temp_cols_with_suff_data

#### =======================
def select_visits_with_dual_thresholds(
    df, time_col, patient_col, cutoff_day, before_threshold, after_threshold,verbose=True):
    """
    Filters visits for each patient based on a time cutoff with thresholds before and after.
    This way we can adjust for patients, who visited the clinic a couple of days later than scheduled

    Logic:
    - If the visit is before the cutoff and lies within the before-threshold, keep all visits up to that visit:
    - If there are no post-cutoff visits, take all visits up to the last prior visit, doesn't matter if it lies within the before-threshold or not. 
    - If there are post-cutoff visits, but the last visit before cutoff lies outside of the before-threshold, take all visits up to the post-cutoff visit, 
        if the post-cutoff visit lies within the after threshold.
    
    Parameters:
        df (pd.DataFrame): Input data with multiple visits per patient.
        time_col (str): Column name containing visit times (numeric).
        patient_col (str): Column name identifying patients.
        cutoff_day (int or float): Time cutoff.
        before_threshold (int or float): Max days before cutoff to accept a visit.
        after_threshold (int or float): Max days after cutoff to accept a visit.
    
    Returns:
        filtered_df (pd.DataFrame): Filtered visits for each patient.
    """
    selected_visits = []
    excluded_patients = []

    for patient_id, group in df.groupby(patient_col):
        group_sorted = group.sort_values(time_col)
        group_sorted = group_sorted.copy()
        group_sorted["diff"] = group_sorted[time_col] - cutoff_day

        before = group_sorted[(group_sorted["diff"] <= 0)]
        after = group_sorted[(group_sorted["diff"] > 0)]

        # Case 1: visit before cutoff within threshold
        valid_before = before[before["diff"] >= -before_threshold]
        if not valid_before.empty:
            last_before_day = valid_before[time_col].max()
            keep = group_sorted[group_sorted[time_col] <= last_before_day]
            selected_visits.append(keep)
            continue

        # Case 2: no valid_before, but valid after
        valid_after = after[after["diff"] <= after_threshold]
        if not valid_after.empty:
            first_after_day = valid_after[time_col].min()
            keep = group_sorted[group_sorted[time_col] <= first_after_day]
            selected_visits.append(keep)
            continue

        # Case 3: no valid_before and no valid_after, but some visits before
        if not before.empty:
            last_before_day = before[time_col].max()
            keep = group_sorted[group_sorted[time_col] <= last_before_day]
            selected_visits.append(keep)
        else:
            excluded_patients.append(patient_id)

    filtered_df = pd.concat(selected_visits, axis=0).drop(columns='diff')

    if verbose:
        total_patients = df[patient_col].nunique()
        num_excluded_patients=len(excluded_patients)
        print(f"Excluded {num_excluded_patients} out of {total_patients} patients "
              f"({num_excluded_patients / total_patients:.1%})\n")
    
    return filtered_df

In [82]:
merged_df = load_merged_data_of_lab_vars()

In [87]:
merged_df.loc[:,merged_df.columns[merged_df.columns.str.contains('AccuProbe')]].dropna(how='all',axis=0)

#merged_df

mb_AccuProbe_STD_RESULT  mb_AccuProbe_STD_CAT_RESULT  \
9971                 positive                          NaN   
9986                 positive                          NaN   
9997                 positive                          NaN   
10028                positive                          NaN   
10029                positive                          NaN   
...                       ...                          ...   
41790                positive                          NaN   
41791                positive                          NaN   
41792                positive                          NaN   
41794                positive                          NaN   
41802                positive                          NaN   

       mb_AccuProbe_CULTURE_STATUS  mb_AccuProbe_STD_NUM_UNIT  \
9971                           NaN                        NaN   
9986                           NaN                        NaN   
9997                           NaN                        NaN   
10028                          NaN                        NaN   
10029                          NaN                        NaN   
...                            ...                        ...   
41790                          NaN                        NaN   
41791                          NaN                        NaN   
41792                          NaN                        NaN   
41794                          NaN                        NaN   
41802                          NaN                        NaN   

       mb_AccuProbe_STD_CAT_ORDINAL_RESULT  mb_AccuProbe_STD_NUM_RESULT  
9971                                   NaN                          NaN  
9986                                   NaN                          NaN  
9997                                   NaN                          NaN  
10028                                  NaN                          NaN  
10029                                  NaN                          NaN  
...                                    ...                          ...  
41790                                  NaN                          NaN  
41791                                  NaN                          NaN  
41792                                  NaN                          NaN  
41794                                  NaN                          NaN  
41802                                  NaN                          NaN  

[3269 rows x 6 columns]

# Load data

## Run this for first time running this script on a new dataset

In [7]:
keep_data_with_unknown_drug_regimen=False
keep_days_with_lab_measurements_only=True

variables_to_load='all'

for key in [*parameters_for_analysis][:1]:
    print(key)
    parameter_dict=parameters_for_analysis[key]
    result_cat=parameter_dict['result_cat']
    selection_method=parameter_dict['selection_method']
    clust_comb=parameter_dict['clust_comb']
    num_of_common_vars=parameter_dict['num_of_common_vars']
    graph_metric=parameter_dict['graph_metric']


    print('selection method: ',key)
    common_variables_for_analysis,pats_for_analysis = return_common_vars_pats_for_anal(result_cat,selection_method,
                                                                                      num_of_common_vars,clust_comb,
                                                                                      graph_metric)

    ## To subset the "measured" variables to the PRE-SELECTED COMMON VARS ACROSS PATIENTS in s7_2.notebook 
    #  (here stored as "num_of_common_vars" variable) => run select_columns_for_analysis
    if variables_to_load=='pre-selected':
        variables_for_analysis=common_variables_for_analysis

    ## To select ALL MEASURED variables that were measured in the analysis time period, run get_all_var_names
    if variables_to_load=='all':
        _,all_vars=get_all_var_names()
        variables_for_analysis=all_vars
    
    
    data_conc = concatenate_temporal_data(pats_for_analysis,keep_data_with_unknown_drug_regimen,
                                         variables_for_analysis,
                                         keep_days_with_lab_measurements_only)
    
    data_conc_cols_subset = subset_columns_relevant_for_analysis(data_conc,pats_for_analysis,variables_for_analysis,
                                                                 return_selected_columns_only=True)

    ## Drop columns with all 0 values
    data_conc=data_conc.loc[:,~data_conc.apply(lambda x: all(x==0))]
    
      
    fn='../data/'+key+'_all_data_concat.csv.gz'
    data_conc.to_csv(fn,compression='gzip')
    #del data_conc

tb21_22_2984_pats_22_vars_result_at_end_of_treatment
selection method:  tb21_22_2984_pats_22_vars_result_at_end_of_treatment
lab measurement data: done  0.08  minutes
dataframe shape:  (61404, 969)
dr_reg done  0.43  minutes
dataframe shape:  (37616, 981)
ae_temp done  1.1  minutes
dataframe shape:  (37616, 2370)
cm_temp_drugs_doses done  1.19  minutes
dataframe shape:  (37616, 2866)
cm_temp_drugs_days_of_appl done  1.36  minutes
dataframe shape:  (37616, 3578)
cm_temp_ind done  1.54  minutes
dataframe shape:  (37616, 4106)
mh done  1.56  minutes
dataframe shape:  (37616, 5146)


## Check number of patients with data entries for each dataset type

In [5]:
ds_type='ms_'
ds_type='ae'

all_vars_dict,all_vars=get_all_var_names()
[*all_vars_dict][13:14]

for ds_type in [*all_vars_dict][12:13]:
    a=data_conc.loc[:,['USUBJID','STUDYID','DAY'] + data_conc.columns[data_conc.columns.str.startswith(ds_type)].tolist()].dropna(subset=data_conc.columns[data_conc.columns.str.startswith(ds_type)].tolist(),how='all',axis=0)
    #print(data_conc.shape)
    print(f'++{ds_type}++ \nvars available for unique patients:\n',a.groupby('STUDYID').apply(lambda x:x['USUBJID'].unique().shape),'\n')

NameError: name 'get_all_var_names' is not defined

In [169]:
temp_col_threshold=0.005 # 0.3 
cols_to_keep=select_temporal_cols_with_suff_pat_data('ae',data_conc,temp_col_threshold)
(cols_to_keep)


c=a[['USUBJID','STUDYID'] + cols_to_keep].dropna(subset=cols_to_keep,how='all',axis=0)
print(f'++{ds_type}++ \nvars available for unique patients:\n',c.groupby('STUDYID').apply(lambda x:x['USUBJID'].unique().shape),'\n')

['ae_HAEMOPTYSIS', 'ae_JOINT PAIN', 'ae_HEADACHE', 'ae_DIARRHEA', 'ae_RHINITIS', 'ae_ELEVATED ALT', 'ae_TACHYCARDIA', 'ae_ELEVATED AST', 'ae_FEVER', 'ae_COUGH', 'ae_ELEVATED LIVER ENZYMES', 'ae_RASH', 'ae_ITCHING', 'ae_CONSTIPATION', 'ae_LOSS OF APPETITE', 'ae_ELEVATED CREATININE', 'ae_ANAEMIA', 'ae_DYSPNEA', 'ae_PROLONGED PTT', 'ae_NAUSEA', 'ae_VOMITING', 'ae_ORAL CANDIDIASIS', 'ae_CHEST PAIN', 'ae_WEIGHT LOSS', 'ae_INFLUENZA', 'ae_NIGHT SWEATS', 'ae_MALARIA', 'ae_PERIPHERAL NEUROPATHY', 'ae_RESPIRATORY ABNORMALITIES', 'ae_UPPER RESPIRATORY TRACT INFECTION', 'ae_HYPERGLYCEMIA', 'ae_HYPOGLYCEMIA', 'ae_INCREASED AMYLASE', 'ae_DIZZINESS', 'ae_ABDOMINAL PAIN', 'ae_CERVICAL LYMPHADENOPATHY', 'ae_WEAKNESS', 'ae_LEUCOPENIA', 'ae_COMMON COLD', 'ae_HYPOKALAEMIA', 'ae_HYPERKALAEMIA', 'ae_LYMPHADENOPATHY', 'ae_LEUCOCYTOSIS', 'ae_KIDNEY FAILURE', 'ae_HYPERCREATINEMIA', 'ae_NPN INCREASED', 'ae_QTC PROLONGATION', 'ae_PARASITOSIS']
++ae++ 
vars available for unique patients:
 STUDYID
TB-1021     (83

# Convert tabular data to text

## Functions

In [5]:
## Export the dataframe to a dictionary where the first keys are the patients, the second are the days, 
#  3rd layer are the name of the mb testing mwthods with the results creaeted previously as strings
import warnings
warnings.filterwarnings('ignore')


def convert_df_to_dict_format(merged_df):
    a=merged_df.groupby(['USUBJID'],group_keys=True).apply(lambda x: x.set_index('DAY').to_dict(orient='index')).to_dict()

    ### CONVERT THE DICTIONARY OF RESULTS INTO A STRING INPUT THAT CAN BE FED TO AN LLM 
    ## Create dictionary holding the result of MB tresting from one day as a string: 
    #  {pat_id (str): day of study (str):{vs_measurement1:result of vs_measurement1; 
    #                                            vs_measurement1:result of vs_measurement1;... (str)}}
    string_dict={}

    for pat in [*a][0:]:
        string_dict[pat]={}
        for day in [*a[pat]]:
             
            str_list=[f'{method}: {result}' for method,result in a[pat][day].items() if str(result) not in ['nan','Nan']]
      
            day_string='; '.join(str_list)   
            string_dict[pat][f'Day {day}']=day_string
  
    return string_dict

### =================
def convert_dm_vars(df,ds_type_var_colnames,ds_type):

    ll=['STUDYID','ARM','DAY']
    for coln in ll:
        if coln in df.columns:
            df=df.drop(columns=[coln])
        
    df=df.drop_duplicates()
    df=df.set_index('USUBJID')

    ## Capitalize string of column names and entries in df
    df['SEX']=df['SEX'].replace({'M':'Male','F':'Female',1:'Male',0:'Female'})
    df.loc[~df['AGE'].isna(),'AGE']=df.loc[~df['AGE'].isna(),'AGE'].astype(int).astype(str)
    
    if 'RACE' in df.columns:
        df['RACE']=df['RACE'].str.capitalize()

    df=df.dropna(how='all',axis=0)
    
    df.columns=df.columns.str.capitalize()
    df_dict=df.T.to_dict()
    return df_dict


### =================
def convert_mb_vars(df,ds_type_vars,ds_type):
    df=df.dropna(how='all',axis=0)
    
    ## Drop duplicated columns
    df = df.loc[:,~df.columns.duplicated()].copy()

    ## Where MGIT is negative, delete the 43.0 days ==> I put the 43 days there for regression prediction
    if 'mb_MGIT_STD_RESULT' in df.columns:
        df.loc[df['mb_MGIT_STD_RESULT']=='negative',['mb_MGIT_STD_NUM_RESULT','mb_MGIT_STD_NUM_UNIT']]=np.nan

    ds_type_vars = list(set([ds_type_var for ds_type_var in ds_type_vars for coln in df.columns if coln.startswith(ds_type_var)]))

    print(ds_type_vars)
    l=[]

    # For each mb testing method (type_var here) collect the results into one column as a string for each visit
    for type_var in ds_type_vars[:]: #ds_type_vars:
    #for type_var in ['mb_MGIT']:
        #print(type_var)
        #[True if coln.startswith(type_var) else False for coln in df.columns]
        
        ## Create variable name by dropping type_var prefix & capitalization 
        #  i.e. mb_Time to Detection => Time to detection
        res_colname=type_var.split(f'{ds_type}_')[-1].capitalize()

        num_colnames=[f'{type_var}_STD_NUM_RESULT',f'{type_var}_STD_NUM_UNIT']
        cat_colnames=[f'{type_var}_STD_CAT_RESULT']
        
        ## Don't consider culture_status at baseline
        if end_day=='baseline':
            df_type_var=df[num_colnames+[f'{type_var}_STD_RESULT',f'{type_var}_STD_CAT_RESULT','DAY','USUBJID']]#.dropna(subset=num_colnames)
        if end_day!='baseline':
            df_type_var=df[num_colnames+[f'{type_var}_STD_RESULT',f'{type_var}_CULTURE_STATUS',f'{type_var}_STD_CAT_RESULT','DAY','USUBJID']]#.dropna(subset=num_colnames)
        df_type_var=df_type_var.set_index(['DAY','USUBJID'],drop=True)

        ## Create filter for selecting rows with only numerical results
        #num_filt=~df[num_colnames].isna().all(axis=1).values
        #num_filt=~df[num_colnames].isna().all(axis=1).values
        num_filt =~df[f'{type_var}_STD_NUM_RESULT'].isna().values


        ## Add new column to collect mb test method: result as string
        df_type_var[res_colname]=np.nan

        ## IF THERE ARE NUMERICAL RESULTS. AVAILABLE, ADD THEM IN FRONT OF THE STD RESULT
        ## For numerical add the numerical result + categorical as well (i.e. TTP: 13.2 Days, positive)
        df_type_var.loc[num_filt,res_colname]=df_type_var.loc[num_filt,f'{type_var}_STD_NUM_RESULT'].astype(str).str.capitalize() + ' '+\
                                                            df_type_var.loc[num_filt,f'{type_var}_STD_NUM_UNIT'].astype(str).str.capitalize()+ ', '+\
                                                            df_type_var.loc[num_filt,f'{type_var}_STD_RESULT'].astype(str).str.capitalize()

        ## IF THERE ARE CATEGORICAL ENTRIES AVAIABLE, ADD THEM TO THE RESULT
        if f'{type_var}_STD_CAT_RESULT' in df.columns:
            cat_filt=~df[cat_colnames].isna().all(axis=1).values
            df_type_var.loc[cat_filt,res_colname]=df_type_var.loc[cat_filt,f'{type_var}_STD_CAT_RESULT'].astype(str).str.capitalize()#+ ', '+\
                                                                #df_type_var.loc[cat_filt,f'{type_var}_STD_RESULT'].astype(str).str.capitalize()


        ## IF THERE ARE NO CATEGORICAL OR NUMERICAL ENTRIES AVAIABLE, ONLY THE STANARDISED RESULT 
        ## ==> ADD STANDARD TEST RESULT (positive/negative) ==> don't convert to str yet, as this columns has np.Nans, that we drop later
        #no_cat_num_filt=df[cat_colnames+num_colnames].isna().all(axis=1).values
        no_cat_num_filt=df[[f'{type_var}_STD_NUM_RESULT',f'{type_var}_STD_CAT_RESULT']].isna().all(axis=1).values
        df_type_var.loc[no_cat_num_filt,res_colname]=df_type_var.loc[no_cat_num_filt,f'{type_var}_STD_RESULT']#.astype(str).str.capitalize()

        ## If variable also has. a culture status entry, add that as a separate column
        #. ==> don't convert to string either yet, as this column as contains NaNs
        non_num_colnames_to_add=[res_colname]
        if f'{type_var}_CULTURE_STATUS' in df.columns:
            df_type_var[f'{res_colname} culture status']=np.nan #'Nan'
            
            filt_=~df[f'{type_var}_STD_RESULT'].isna().values         
            df_type_var.loc[filt_,f'{res_colname} culture status']=df_type_var.loc[filt_,f'{type_var}_CULTURE_STATUS']
            
            
        ## Drop rows (visits), that are all NaNs
        df_type_var=df_type_var[~(df_type_var[num_colnames + non_num_colnames_to_add].isna().all(axis=1))]

        ## If variable doesn't have valid entries, skip to next variable
        if df_type_var.shape[0]==0:
            #print('empty df',type_var)
            continue

        #print(df_type_var)

        ## If culutre status column is in df_type_var, add it to the non-num columns, next to the STD_RESULT column of the variable
        # ==> this means the variable has valied entries in the culture status column
        if f'{res_colname} culture status' in df_type_var.columns:
            non_num_colnames_to_add.append(f'{res_colname} culture status')

        #df_type_var=df_type_var.astype(str)
        #print(res_colname,df_type_var)
          
        ## Append to a list for concatenation later
        #l.append(df_type_var[res_colname])
        l.append(df_type_var[non_num_colnames_to_add])
        


    ## Merge the dataframes from all the testing methods into one
    merged_df=pd.concat(l,axis=1)  
    #merged_df=merged_df.loc[:,~(merged_df=='Nan').all()]

    
    ## Rename come columns and drop nan values from the string
    replace_dict={'Lj-culture':'LJ-slope',
                  'Mgit':'MGIT'}
    for key,value in replace_dict.items():
        merged_df.columns=merged_df.columns.str.replace(key,value,regex=True)

    #merged_df=merged_df.replace({'nan':''},regex=True) 

    ## Set USUBJID as index for the creation of a dictionary holding all the results for each patient
    merged_df=merged_df.reset_index(drop=False).set_index('USUBJID')
    merged_df=merged_df.sort_values(by='DAY')
    merged_df['DAY']=merged_df['DAY'].astype(int)

    #return merged_df
    
    ## Convert dataframe to dictionary
    string_dict=convert_df_to_dict_format(merged_df)
  
    return string_dict
    
### =================
def convert_vs_vars(df,ds_type_var_colnames,ds_type):

    ## Drop duplicated columns
    df = df.loc[:,~df.columns.duplicated()].copy()
    l=[]

    # For each vs measurement (type_var here) collect the results into one column as a string for each visit
    for type_var in ds_type_vars:
        
        ## Create variable name by dropping type_var prefix & capitalization 
        #  i.e. vs_Blood Pressure => Blood Pressure
        res_colname=type_var.split(f'{ds_type}_')[-1].capitalize()
        
        num_colnames=[f'{type_var}_STD_NUM_RESULT',f'{type_var}_STD_NUM_UNIT']
        df_type_var=df[num_colnames+['DAY','USUBJID']].dropna(subset=num_colnames)
        df_type_var=df_type_var.set_index(['DAY','USUBJID'],drop=True)

        ## Add new column to collect vs measurement result as string
        df_type_var[res_colname]=np.nan

        #print(df_type_var.loc[:,[f'{type_var}_STD_NUM_RESULT',f'{type_var}_STD_NUM_UNIT']])
        
        ## For numerical add the numerical result + categorical as well (i.e. TTP: 13.2 Days, positive)
        df_type_var.loc[:,res_colname]=df_type_var.loc[:,f'{type_var}_STD_NUM_RESULT'].astype(str).str.capitalize() + ' '+\
                                                            df_type_var.loc[:,f'{type_var}_STD_NUM_UNIT'].astype(str)#.str.capitalize() 
        

        df_type_var=df_type_var[~(df_type_var[num_colnames].isna().all(axis=1))]
          
        ## Append to a list for concatenation later
        l.append(df_type_var[res_colname])


    ## Merge the dataframes from all the testing methods into one
    merged_df=pd.concat(l,axis=1)  
    merged_df=merged_df.dropna(how='all',axis=0)
    merged_df=merged_df.replace({'nan':''},regex=True) 

    ## Set USUBJID as index for the creation of a dictionary holding all the results for each patient
    merged_df=merged_df.reset_index(drop=False).set_index('USUBJID')
    merged_df=merged_df.sort_values(by='DAY')
    merged_df['DAY']=merged_df['DAY'].astype(int)

    string_dict=convert_df_to_dict_format(merged_df)
  
    return string_dict
    


### =================
def convert_re_vars(df,ds_type_var_colnames,ds_type):
    #print(df[df.columns[df.columns.str.contains('STD_LEFT')]].dropna(how='all',axis=1))
    #df=df.dropna(how='all',axis=1)
    
    ## Drop duplicated columns
    df = df.loc[:,~df.columns.duplicated()].copy()
    df[df.columns[df.columns.str.contains('STD_LEFT')]]=df[df.columns[df.columns.str.contains('STD_LEFT')]].replace({1.0:'Left',0.0:np.nan},regex=True) 
    df[df.columns[df.columns.str.contains('STD_RIGHT')]]=df[df.columns[df.columns.str.contains('STD_RIGHT')]].replace({1.0:'Right',0.0:np.nan},regex=True) 
    df[df.columns[df.columns.str.contains('_STD_BILATERAL')]]=df[df.columns[df.columns.str.contains('_STD_BILATERAL')]].replace({1.0:'Bilateral',0.0:np.nan},regex=True) 
    df[df.columns[df.columns.str.contains('_STD_UNILATERAL')]]=df[df.columns[df.columns.str.contains('_STD_UNILATERAL')]].replace({1.0:'Unilateral',0.0:np.nan},regex=True) 

    l=[]
    
    # For each vs measurement (type_var here) collect the results into one column as a string for each visit
    for type_var in ds_type_vars:
        
        ## Create variable name by dropping type_var prefix & capitalization 
        #  i.e. re_Zone Score => Zone score
        res_colname=type_var.split(f'{ds_type}_')[-1].capitalize()
        
        side_colnames=[f'{type_var}_STD_LEFT',f'{type_var}_STD_RIGHT',f'{type_var}_STD_UNILATERAL',f'{type_var}_STD_BILATERAL'] #
        df_type_var=df.dropna(subset=f'{type_var}_STD_CAT_RESULT')
        df_type_var=df_type_var.set_index(['DAY','USUBJID'],drop=True)

        ## Add new column to collect measurement result as string
        df_type_var[res_colname]=np.nan

        
        ## Create filter for selecting rows where side information is missing (if all side info is missing==> True)
        side_colnames_present=list(set(side_colnames)& set(df_type_var.columns))
        side_missing_filt=df_type_var[side_colnames_present].isna().all(axis=1)
        

        ## Convert Zone score to integer
        if type_var=='re_Zone Score':
            df_type_var.loc[:,f'{type_var}_STD_CAT_RESULT']=df_type_var.loc[:,f'{type_var}_STD_CAT_RESULT'].astype(int)
        
   
        ## If no side information available, extract result and inidicate missing side information
        if side_missing_filt.all()==True:
            df_type_var.loc[:,res_colname]=df_type_var.loc[:,f'{type_var}_STD_CAT_RESULT'].astype(str).str.capitalize()\
                                                                           #+ ', Side unknown' 
        ## If any of the side columns contains information, check which one (left-right or uni/bilateral)
        if side_missing_filt.all()==False:
            left_right_colnames=[f'{type_var}_STD_LEFT',f'{type_var}_STD_RIGHT']
            lateral_colnames=[f'{type_var}_STD_UNILATERAL',f'{type_var}_STD_BILATERAL']
            left_right_missing_filt=df_type_var[left_right_colnames].isna().all(axis=1)
            

            ## If there is exact information on the left-right sidedness, add that as a suffix to the result
            filt=~left_right_missing_filt.values
            df_type_var.loc[filt,res_colname]=df_type_var.loc[filt,f'{type_var}_STD_CAT_RESULT'].astype(str).str.capitalize()\
                                                                          + ', '+ df_type_var.loc[filt,f'{type_var}_STD_LEFT'].astype(str).str.capitalize() \
                                                                            + ', '+ df_type_var.loc[filt,f'{type_var}_STD_RIGHT'].astype(str).str.capitalize()\
                                                                            +' side'

            
            ## If there is ONLY information about uni/bilateral, add that to the result as suffix
            #  This filter filters for rows, where left-right is missing, but there is uni/bilateral information
            lateral_avail_filt=~(df_type_var[lateral_colnames].isna().all(axis=1)) & (df_type_var[left_right_colnames].isna().all(axis=1))
            filt=lateral_avail_filt.values
            df_type_var.loc[filt,res_colname]=df_type_var.loc[filt,f'{type_var}_STD_CAT_RESULT'].astype(str).str.capitalize()\
                                                                          + ', '+ df_type_var.loc[filt,f'{type_var}_STD_UNILATERAL'].astype(str).str.capitalize() \
                                                                            + ', '+ df_type_var.loc[filt,f'{type_var}_STD_BILATERAL'].astype(str).str.capitalize()

                
    
        ## Append to a list for concatenation later
        l.append(df_type_var[res_colname])


    ## Merge the dataframes from all the testing methods into one
    merged_df=pd.concat(l,axis=1)
    merged_df=merged_df.replace({'nan':''},regex=True) 

    ## Change some varibale names to be descriptive, based on TB-1022 study protocol:
    #  https://www.nejm.org/doi/suppl/10.1056/NEJMoa1315817/suppl_file/nejmoa1315817_appendix.pdf
    merged_df.columns=merged_df.columns.str.replace('Bilateral disease','Extent of lung disease',regex=True)
    merged_df.columns=merged_df.columns.str.replace('Zone score','Lung regions affected',regex=True)

    ## Set USUBJID as index for the creation of a dictionary holding all the results for each patient
    merged_df=merged_df.reset_index(drop=False).set_index('USUBJID')
    merged_df=merged_df[~merged_df['DAY'].isna()]
    merged_df=merged_df.sort_values(by='DAY')
    merged_df['DAY']=merged_df['DAY'].astype(int)
    string_dict=convert_df_to_dict_format(merged_df)
  
    return string_dict
    
   
### =================
def convert_lb_vars(df,ds_type_vars,ds_type):
    with open('../data/lab_variables.pkl', 'rb') as f:
        lab_variable_type_dict = pickle.load(f)

    cat_vars=lab_variable_type_dict['lb_categorical_vars']
    num_vars=lab_variable_type_dict['lb_numerical_vars']
    
    if 'lb_Urine pH_STD_NUM_RESULT' in df.columns:
        df['lb_Urine pH_STD_NUM_UNIT']=np.nan
        df.loc[~df['lb_Urine pH_STD_NUM_RESULT'].isna(),'lb_Urine pH_STD_NUM_UNIT']=''
        
    

    ## Drop duplicated columns + empty columns
    df=df.loc[:,~df.columns.duplicated()].copy()
    df=df.dropna(how='all',axis=1)

    ## Filter for lb_variables that haven't been dropped due missingness in all rows
    ds_type_vars=list(set([ds_type_var for coln in df.columns for ds_type_var in ds_type_vars if coln.startswith(ds_type_var)]))
 

    l=[]

    # For each vs measurement (type_var here) collect the results into one column as a string for each visit
    for type_var in ds_type_vars:

        ## Create variable name by dropping type_var prefix & capitalization 
        #  i.e. lb_Blood Pressure => Blood Pressure
        res_colname=type_var.split(f'{ds_type}_')[-1].capitalize()
        
        if type_var.split(f'{ds_type}_')[-1] in cat_vars:
            
            cat_colnames=[f'{type_var}_STD_CAT_RESULT']
            df_type_var=df[cat_colnames+['DAY','USUBJID']].dropna(subset=cat_colnames)
            df_type_var=df_type_var.set_index(['DAY','USUBJID'],drop=True)

            print(df_type_var)

            ## For numerical add the numerical result + categorical as well (i.e. TTP: 13.2 Days, positive)
            df_type_var.loc[:,res_colname]=df_type_var.loc[:,f'{type_var}_STD_CAT_RESULT'].astype(str).str.capitalize()
            
    
        if type_var.split(f'{ds_type}_')[-1] in num_vars:
            num_colnames=[f'{type_var}_STD_NUM_RESULT',f'{type_var}_STD_NUM_UNIT']
            df_type_var=df[num_colnames+['DAY','USUBJID']].dropna(subset=num_colnames)
            #filt=df_type_var[num_colnames].isna().all(axis=1)
            df_type_var=df_type_var.set_index(['DAY','USUBJID'],drop=True)
                
            
            ## For numerical add the numerical result + categorical as well (i.e. TTP: 13.2 Days, positive)
            df_type_var.loc[:,res_colname]=df_type_var.loc[:,f'{type_var}_STD_NUM_RESULT'].round(2).astype(str).str.capitalize() + ' '+\
                                                            df_type_var.loc[:,f'{type_var}_STD_NUM_UNIT'].astype(str).str.lower() 
        

        df_type_var=df_type_var[~(df_type_var[num_colnames].isna().all(axis=1))]
          
        ## Append to a list for concatenation later
        l.append(df_type_var[res_colname])


    ## Merge the dataframes from all the testing methods into one
    merged_df=pd.concat(l,axis=1)  
    merged_df=merged_df.dropna(how='all',axis=0)
    merged_df=merged_df.replace({'nan':''},regex=True) 

    
    ## Set USUBJID as index for the creation of a dictionary holding all the results for each patient
    merged_df=merged_df.reset_index(drop=False).set_index('USUBJID')
    merged_df=merged_df.sort_values(by='DAY')
    merged_df['DAY']=merged_df['DAY'].astype(int)

    string_dict=convert_df_to_dict_format(merged_df)
  
    return string_dict
    

### =================
def convert_dr_reg_vars(df,ds_type_var_colnames,ds_type):

    ## Drop duplicated columns
    df = df.loc[:,~df.columns.duplicated()].copy()

    ## Drop those days, where drug was not taken anymore (only relevant in sequential input)
    if data_inclusion_type in ['baseline_vars','all_days']:
        df = df.dropna(how='all',subset=df.columns[df.columns.str.contains('dr_reg')].tolist(),axis=0)
        
    l=[]

    df=df[df['DAY']>-1]
    #print(df.sort_values(by=['USUBJID','DAY']))
    #print(df['DAY'].sort_values().unique())
    # For each vs measurement (type_var here) collect the results into one column as a string for each visit
    for type_var in ds_type_vars:
        #print(type_var)
        
        ## Create variable name by dropping type_var prefix & capitalization 
        #  i.e. vs_Blood Pressure => Blood Pressure
        if type_var!='dr_reg_study_drugs_cumul':
            type_var_coln=f'{type_var}_cumulative_dose'
            #res_colname=type_var.split(f'{ds_type}_')[-1].capitalize() #+ ' cumulative dose'
            #res_colname=type_var.split(f'{ds_type}_')[-1].capitalize()+' cumulative dose'
            res_colname=type_var.split(f'{ds_type}_')[-1].capitalize()#+' cumulative dose'
            
            ## Drop patients who didn't take given drug
            pats_not_taking_drug=df.groupby('USUBJID').apply(lambda x: max(x[type_var_coln])>0)
            pat_filt=pats_not_taking_drug[pats_not_taking_drug].index.tolist()
        
        if type_var=='dr_reg_study_drugs_cumul':
            type_var_coln=f'{type_var}'
            res_colname='Days drug taken'
            
            ## Drop patients who didn't take given drug
            pats_not_taking_drug=df.groupby('USUBJID').apply(lambda x: max(x[type_var_coln])>0)
            pat_filt=pats_not_taking_drug[pats_not_taking_drug].index.tolist()

 
        df_type_var=df.loc[df['USUBJID'].isin(pat_filt),:].set_index(['DAY','USUBJID'],drop=True)

        ## Add new column to collect vs measurement result as string
        df_type_var[res_colname]=np.nan

        if type_var!='dr_reg_study_drugs_cumul':
            
            if 'placebo' not in type_var:
            ## For numerical add the numerical result + categorical as well (i.e. TTP: 13.2 Days, positive)
            #df_type_var.loc[:,res_colname]=(df_type_var.loc[:,type_var_coln]/1000).astype(str).str.capitalize() + ' mg'
                df_type_var[res_colname]=(df_type_var[type_var_coln]).astype(str).str.capitalize() + ' mg'
                #print('after string\n',df_type_var[res_colname],'\n')
                df_type_var[res_colname] = df_type_var[res_colname].replace('.0 mg',' mg',regex=True).values
                #print('after replace\n',df_type_var[res_colname],'\n')
            if 'placebo' in type_var:
                df_type_var[res_colname]=(df_type_var[type_var_coln]).astype(str).str.capitalize().replace(r'\.0$', '', regex=True)
                
                
        if type_var=='dr_reg_study_drugs_cumul':
            df_type_var[res_colname]=df_type_var[type_var_coln].astype(str).replace(r'\.0$', '', regex=True)
            #df_type_var[res_colname] = df_type_var[res_colname].replace('.0','',regex=True).values
            
        ## Append to a list for concatenation later
        l.append(df_type_var[res_colname])


    ## Merge the dataframes from all the testing methods into one
    merged_df=pd.concat(l,axis=1)  
    merged_df=merged_df.dropna(how='all',axis=0)
    merged_df=merged_df.replace({'nan':'','Nan':''},regex=True) 
    #print(merged_df)
    
    ## Set USUBJID as index for the creation of a dictionary holding all the results for each patient
    merged_df=merged_df.reset_index(drop=False).set_index('USUBJID')
    merged_df=merged_df.sort_values(by='DAY')
    merged_df['DAY']=merged_df['DAY'].astype(int)

    string_dict=convert_df_to_dict_format(merged_df)
  
    return string_dict


### =================
def convert_mh_vars(df,ds_type_var_colnames,ds_type):

    ## Drop duplicated columns
    df = df.loc[:,~df.columns.duplicated()].copy()
    #df=df.drop_duplicates(subset=['USUBJID'])

    df=df.dropna(how='all',axis=1)

    ## For each patient, extract if there is non-NaN value mh variable available, registered at any time point in the data
    df=df.groupby('USUBJID').apply(lambda x: ~(x.loc[:,x.columns.str.startswith('mh')].isna().any(axis=0)))

    
    ## For each patient, set available mh variables to True, and if variable was not recorded, set to NaN
    #b=df.reset_index().groupby('USUBJID').apply(lambda x: x[x==True].dropna(axis=1,how='any')).reset_index().drop(columns=['level_1']).set_index('USUBJID')
    b=df.replace(False, np.nan)
    
    ## Drop mh_ from the beginning of the variable name
    b.columns=[col.split('mh_')[-1].capitalize() for col in b.columns]

    ## Keep only medical history terms, that have at least thr number of patients who have 'YES' entries for that term
    if data_inclusion_type!='all_days':
        thr=b.shape[0]*0.3
    if data_inclusion_type=='all_days':
        thr=b.shape[0]*0.1
        
    b=b[b.sum(axis=0)[b.sum(axis=0)>thr].index]
        
    #print(b.columns)
    
    #### Create dictionary holding only the mh variables, that patient has a record on having before beginning of the study
    #mh_dict={pat:(b.loc[pat,:].dropna()).index.tolist() for pat in b.index}

    ## Create dictionary with each selected mh variable, and indicating if present or not in medical history of patient
    b_ = b.fillna(False).replace({True:'Yes',False:'No'})
    mh_dict = dict(zip(b_.index.tolist(),[b_.loc[pat,:].to_dict() for pat in b_.index]))

    return mh_dict

### =================
def convert_ce_vars(df,ds_type_var_colnames,ds_type):

    ## Drop duplicated columns
    df = df.loc[:,~df.columns.duplicated()].copy()
    #df=df.drop_duplicates(subset=['USUBJID'])

    #df=df.dropna(how='all',axis=1)

    ## Drop tocgrade if baseline variables are used, as they are also dropped in the baseline raw models
    if data_inclusion_type in ['baseline_vars','baseline_last_day']:
        df.loc[:,df.columns.str.endswith('STD_CETOXGR')]=np.nan
        

    #print(df)
    l=[]
    
    # For each vs measurement (type_var here) collect the results into one column as a string for each visit
    ds_type_vars = [coln.split('_STD_CAT_ORDINAL_RESULT')[0] for coln in df.columns[df.columns.str.startswith('ce')] if '_STD_CAT_ORDINAL_RESULT' in coln]
    
    for type_var in ds_type_vars:
    #for type_var in ['ce_DYSPNEA']:
        
        #print(type_var)
        
        ## Create variable name by dropping type_var prefix & capitalization 
        #  i.e. vs_Blood Pressure => Blood Pressure
        res_colname=type_var.split(f'{ds_type}_')[-1].capitalize()

            
        num_colnames=[f'{type_var}_STD_CAT_RESULT',f'{type_var}_STD_CETOXGR']
        df_type_var=df[num_colnames+['DAY','USUBJID']].dropna(subset=[f'{type_var}_STD_CAT_RESULT'])
        df_type_var=df_type_var.set_index(['DAY','USUBJID'],drop=True)
        df_type_var=df_type_var.replace({'N':'No','Y':'Yes'},regex=True)
        #print(df_type_var)
        
        ## Add new column to collect vs measurement result as string
        df_type_var[res_colname]=np.nan
        
        ## For numerical add the numerical result + categorical as well (i.e. TTP: 13.2 Days, positive)
        mask=(df_type_var[f'{type_var}_STD_CAT_RESULT']=='Yes').values
        if len(mask)>0:       

            ### Select rows, where toxicity grade is not missing ==> here add toxicity grade as well
            mask_with_tox_grade=((df_type_var[f'{type_var}_STD_CAT_RESULT']=='Yes')&(~df_type_var[f'{type_var}_STD_CETOXGR'].isna())).values

            #print(df_type_var.loc[mask_with_tox_grade,:])
            if (mask_with_tox_grade.sum())>0:  
                #print(mask_with_tox_grade)
                df_type_var.loc[mask_with_tox_grade,res_colname] = df_type_var.loc[mask_with_tox_grade,f'{type_var}_STD_CAT_RESULT'].astype(str).str.capitalize() + ', grade '+\
                                                                                        df_type_var.loc[mask_with_tox_grade,f'{type_var}_STD_CETOXGR'].astype(str).str.split('.',expand=True)[0]#.str.capitalize() 


            ### For rows where toxicity grade ismissing, only add the day of the clinical event
            mask_wo_tox_grade=((df_type_var[f'{type_var}_STD_CAT_RESULT']=='Yes')&(df_type_var[f'{type_var}_STD_CETOXGR'].isna())).values

            if (mask_wo_tox_grade.sum())>0:  
                df_type_var.loc[mask_wo_tox_grade,res_colname] = df_type_var.loc[mask_wo_tox_grade,f'{type_var}_STD_CAT_RESULT'].astype(str).str.capitalize()
                                                                                          
            
            df_type_var=df_type_var[~(df_type_var[num_colnames].isna().all(axis=1))]

            
            ## Append to a list for concatenation later
            l.append(df_type_var[res_colname])

        
        mask=(df_type_var[f'{type_var}_STD_CAT_RESULT']=='No').values
        if len(mask)>0:
            df_type_var.loc[mask,res_colname] = df_type_var.loc[mask,f'{type_var}_STD_CAT_RESULT'].astype(str).str.capitalize() 
        
            df_type_var=df_type_var[~(df_type_var[num_colnames].isna().all(axis=1))]
          
            ## Append to a list for concatenation later
            l.append(df_type_var[res_colname])
                   
        

    ## Merge the dataframes from all the testing methods into one
    merged_df=pd.concat(l,axis=1)  
    merged_df=merged_df.dropna(how='all',axis=0)
    merged_df=merged_df.replace({'nan':''},regex=True) 

    #print(merged_df)

    ## Set USUBJID as index for the creation of a dictionary holding all the results for each patient
    merged_df=merged_df.reset_index(drop=False).set_index('USUBJID')
    merged_df=merged_df.sort_values(by='DAY')
    merged_df['DAY']=merged_df['DAY'].astype(int)

    string_dict=convert_df_to_dict_format(merged_df)

    return string_dict


### =================
def convert_cmdos_vars(df,ds_type_var_colnames,ds_type):

    ## Drop duplicated columns
    df = df.loc[:,~df.columns.duplicated()].copy()
    #df=df.drop_duplicates(subset=['USUBJID'])

    df=df.dropna(how='all',axis=1)


    l=[]
    
    df=df[df['DAY']>-1]
    #print(df.sort_values(by=['USUBJID','DAY']))
    #print(df['DAY'].sort_values().unique())
    # For each vs measurement (type_var here) collect the results into one column as a string for each visit

    ds_type_vars= list(set([coln for coln in df.columns[df.columns.str.startswith('cmdos_')] if 'cumul' in coln]))

    df=df.loc[:,['USUBJID','DAY']+ [coln for coln in ds_type_vars]]

    #print(df)
    
    for type_var in ds_type_vars[:]:
        #print(type_var)
        
        ## Create variable name by dropping type_var prefix & capitalization 
        #  i.e. vs_Blood Pressure => Blood Pressure
        #type_var_coln=f'{type_var}_cumulative_dose'
        res_colname=type_var.split('_cumul')[0].split('cmdos_')[-1]#.capitalize() #+ ' cumulative dose'
        #application_route=res_colname.split('_')[-1]
        res_colname=res_colname.replace('_',' ').capitalize()
        res_colname=res_colname.replace('intravenous','IV').capitalize()
        res_colname=res_colname.replace('intramuscular','IM').capitalize()
        res_colname = res_colname + ' cumulative dose'

        #print(res_colname)#,application_route)

        ## Drop patients who didn't take given drug
        pats_not_taking_drug=df.groupby('USUBJID').apply(lambda x: max(x[type_var])>0)
        pat_filt=pats_not_taking_drug[pats_not_taking_drug].index.tolist()
        
        df_type_var=df.loc[df['USUBJID'].isin(pat_filt),['DAY','USUBJID',type_var]].set_index(['DAY','USUBJID'],drop=True)

        #if len(df_type_var)>0:
        #    print(df_type_var)

        df_type_var=df_type_var.dropna(how='all',axis=0)

        ## Add new column to collect vs measurement result as string
        df_type_var[res_colname]=np.nan

        #if type_var!='dr_reg_study_drugs_cumul':
        ## For numerical add the numerical result + categorical as well (i.e. TTP: 13.2 Days, positive)
        #df_type_var.loc[:,res_colname]=(df_type_var.loc[:,type_var]/1000).astype(str).str.capitalize() + ' g'
        df_type_var.loc[:,res_colname]=np.round(df_type_var.loc[:,type_var],0).astype(str).str.capitalize() + ' mg'

            
        ## Append to a list for concatenation later
        l.append(df_type_var[res_colname])


    ## Merge the dataframes from all the testing methods into one
    merged_df=pd.concat(l,axis=1)  
    merged_df=merged_df.dropna(how='all',axis=0)
    merged_df=merged_df.replace({'nan':'','Nan':''},regex=True) 
    #print(merged_df)
    
    ## Set USUBJID as index for the creation of a dictionary holding all the results for each patient
    merged_df=merged_df.reset_index(drop=False).set_index('USUBJID')
    merged_df=merged_df.sort_values(by='DAY')
    merged_df['DAY']=merged_df['DAY'].astype(int)

    string_dict=convert_df_to_dict_format(merged_df)
  
    return string_dict


### =================
def convert_ms_vars(df,ds_type_var_colnames,ds_type):

    ## Drop duplicated columns
    df = df.loc[:,~df.columns.duplicated()].copy()
    #df=df.drop_duplicates(subset=['USUBJID'])

    df=df.dropna(how='all',axis=1)
    l=[]
    
    df=df[df['DAY']>-10]

    # For each vs measurement (type_var here) collect the results into one column as a string for each visit

    #ds_type_vars= list(set([coln for coln in df.columns[df.columns.str.startswith('ms_')] if '_STD_CAT_RESULT' in coln]))
    ds_type_vars = [coln.split('_STD_CAT_RESULT')[0] for coln in df.columns[df.columns.str.startswith('ms_')] if '_STD_CAT_RESULT' in coln]
    
    for type_var in ds_type_vars[:]:
       # print(type_var)
        
        # For each vs measurement (type_var here) collect the results into one column as a string for each visit
        
        ## Create variable name by dropping type_var prefix & capitalization 
        #  i.e. vs_Blood Pressure => Blood Pressure
        res_colname=type_var.split(f'{ds_type}_')[-1].capitalize()
        
        num_colnames=[f'{type_var}_STD_CAT_RESULT']
        df_type_var=df[num_colnames+['DAY','USUBJID']].dropna(subset=num_colnames)
        df_type_var=df_type_var.set_index(['DAY','USUBJID'],drop=True)

        ## Add new column to collect vs measurement result as string
        df_type_var[res_colname]=np.nan

        #print(df_type_var.loc[:,[f'{type_var}_STD_NUM_RESULT',f'{type_var}_STD_NUM_UNIT']])
        
        ## For numerical add the numerical result + categorical as well (i.e. TTP: 13.2 Days, positive)
        df_type_var.loc[:,res_colname]=df_type_var.loc[:,f'{type_var}_STD_CAT_RESULT'].astype(str).str.lower()
        

        df_type_var=df_type_var[~(df_type_var[num_colnames].isna().all(axis=1))]
          
        ## Append to a list for concatenation later
        l.append(df_type_var[res_colname])


    ## Merge the dataframes from all the testing methods into one
    merged_df=pd.concat(l,axis=1)  
    merged_df=merged_df.dropna(how='all',axis=0)
    merged_df=merged_df.replace({'nan':''},regex=True) 

    ## Set USUBJID as index for the creation of a dictionary holding all the results for each patient
    merged_df=merged_df.reset_index(drop=False).set_index('USUBJID')
    merged_df=merged_df.sort_values(by='DAY')
    merged_df['DAY']=merged_df['DAY'].astype(int)


    string_dict=convert_df_to_dict_format(merged_df)
  
    return string_dict


### =================
def convert_ae_vars(df,ds_type_var_colnames,ds_type):

    def label_periods_optimized(df):
        # Convert the list to a NumPy array
        weeks=df['Week'].values
        #days = np.array(days)
    
        # Calculate the differences between consecutive days
        diffs = np.diff(weeks)
    
        # Identify the start of new periods (where the difference is greater than 1)
        new_periods = np.where(diffs > 1)[0] + 1
    
        # Split indices based on detected new periods
        split_indices = np.split(weeks, new_periods)
    
        # Generate labels for each period
        labels = []
        for idx, period in enumerate(split_indices):
            labels.extend([f"{idx + 1}"] * len(period))
    
        return labels

    def extract_severity_and_periods(df):
        # Apply the optimized function to create a new column with period labels
        #print(df)
        if len(df)==1:
            df['Periods']=1
        if len(df)>1:
            df['Periods'] = label_periods_optimized(df)

        ## Extract severity of the event (mode of severity), start and end period of event in weeks 
        df=df.groupby(['Periods','USUBJID'],as_index=True).apply(lambda x: pd.Series({'severity':x[type_var].mode()[0],
                                                                                    'start_week':x['Week'].min(),
                                                                                   'end_week':x['Week'].max()}))
        return df
    

    ## Pre-select the ae terms to be loaded. A there are a lot of terms that are not common across patients, set the number
    #. of patients with entries for goven ae term with 'temp_col_threshold', which sets the threshold at "num_of_patients * temp_col_threshold"
    #. ==> Only those AE terms get selected, that have at occurred in at least "num_of_patients * temp_col_threshold" patients
    
    temp_col_threshold=0.005 # 0.3 
    cols_to_keep=select_temporal_cols_with_suff_pat_data(ds_type,data_conc,temp_col_threshold) 
    
    # List of strings you want to match (e.g., from 'USUBJID' list)
    USUBJID = data_conc[['USUBJID','DAY','STUDYID'] + cols_to_keep].dropna(subset=cols_to_keep,how='all',axis=0)['USUBJID'].unique()
    
    # Read only specific columns
    cols_to_keep_=['USUBJID','DAY','STUDYID'] + cols_to_keep
    
    # Use `chunksize` to read the file in chunks and filter rows for patients in dataset
    filtered_rows = []

    fn=os.path.join('../data','out_ae_standardised_temporal.csv.gz')
    
    for chunk in pd.read_csv(fn, usecols=cols_to_keep_, chunksize=10000):
        # Filter rows where 'USUBJID' column contains values from the `USUBJID` list
        filtered_chunk = chunk[chunk['USUBJID'].isin(USUBJID)]
        filtered_rows.append(filtered_chunk)
    
    # Concatenate the filtered chunks into a single DataFrame
    ae_df = pd.concat(filtered_rows, ignore_index=True)
    
    if isinstance(end_day, str)==False:
        ae_df=ae_df[ae_df['DAY']<=end_day]

    l=[]
    
    for type_var in cols_to_keep[:]:
        #print(type_var)
        
        # For each vs measurement (type_var here) collect the results into one column as a string for each visit
        
        ## Create variable name by dropping type_var prefix & capitalization 
        #  i.e. vs_Blood Pressure => Blood Pressure
        res_colname=type_var.split(f'{ds_type}_')[-1].capitalize()
        
        num_colnames=[f'{type_var}']
        df_type_var=ae_df[num_colnames+['DAY','USUBJID']].dropna(subset=num_colnames)
        df_type_var['Week']=np.floor(df_type_var['DAY']/7)
        df_type_var=df_type_var.set_index(['DAY','USUBJID'],drop=True)
        
        d=df_type_var.groupby(['USUBJID'],as_index=False).apply(lambda x:extract_severity_and_periods(x.sort_values(by=['Week'],ascending=True)))
    
        ## Add new column to collect vs measurement result as string
        d[res_colname]=np.nan

        mask=d['start_week']==d['end_week']
        
        ## Merge results for periods, where event happened only in one week
        d.loc[mask,res_colname]='Week '+ d.loc[mask,'start_week'].astype(int).astype(str) + ', severity ' + d.loc[mask,'severity'].astype(int).astype(str) 
    
        ## Merge results for periods, where event happened only in one week
        d.loc[~mask,res_colname]='Week '+ d.loc[~mask,'start_week'].astype(int).astype(str) + '-' +d.loc[~mask,'end_week'].astype(int).astype(str) +\
                                                        ', severity ' + d.loc[~mask,'severity'].astype(int).astype(str) 
    
        d = d.sort_values(by=['USUBJID','start_week'])
        d=d.reset_index().set_index(['USUBJID','start_week'])
        
        ## Append to a list for concatenation later
        l.append(d[res_colname])
        #week_list.append(d['start_week'])

    ## Merge the dataframes from all the testing methods into one
    merged_df=pd.concat(l,axis=1)  
    merged_df = merged_df.sort_index()
    
    merged_df=merged_df.dropna(how='all',axis=0)
    merged_df=merged_df.replace({'nan':''},regex=True) 
    
    ## Set USUBJID as index for the creation of a dictionary holding all the results for each patient
    a = merged_df.groupby(['USUBJID'],group_keys=False).apply(lambda x: x.to_dict())#.to_dict()
    
    ### CONVERT THE DICTIONARY OF RESULTS INTO A STRING INPUT THAT CAN BE FED TO AN LLM 
    ## Create dictionary holding the result of MB tresting from one day as a string: 
    #  {pat_id (str): day of study (str):{vs_measurement1:result of vs_measurement1; 
    #                                            vs_measurement1:result of vs_measurement1;... (str)}}
    string_dict={}
    for pat in a.index[0:]: #['TB-1021/1104035']
        string_dict[pat]={}
        ae_list=[]
        for ae_event in [*a[pat]]:
            #print(ae_event,a[pat][ae_event])
            #str_list=[f'{ae_event}: {result}' for method,result in a[pat][ae_event].items() if str(result) not in ['nan','Nan']]
            str_list=[f'{result}' for method,result in a[pat][ae_event].items() if str(result) not in ['nan','Nan']]
    
            
            if len(str_list)>0:
                day_string='; '.join(str_list) 
                #print(str_list)
                ae_string=f'{ae_event}: {day_string}'
                ae_list.append(ae_string)
                #print(ae_string)
    
        if len(ae_list)>0:  
            string_dict[pat]='; '.join(ae_list) 
    
    return string_dict


### =================
def convert_su_vars(df,ds_type_var_colnames,ds_type):
    
    ## Drop duplicated columns
    df = df.loc[:,~df.columns.duplicated()].copy()
    #df=df.drop_duplicates(subset=['USUBJID'])

    df=df.dropna(how='all',axis=1)
    
    #df=df[df['DAY']>-10]

    ## For each patient, set available mh variables to True, and if variable was not recorded, set to NaN
    #b=df.reset_index().groupby('USUBJID').apply(lambda x: x[x==True].dropna(axis=1,how='any')).reset_index().drop(columns=['level_1']).set_index('USUBJID')
    b=df.dropna(subset=df.columns[df.columns.str.startswith('su')],how='all',axis=0)
    
    cols_to_drop=['DAY'] + df.columns[df.columns.str.contains('_STD_CAT_ORDINAL_RESULT')].tolist()
    b=b.set_index('USUBJID').drop(columns=cols_to_drop)
    ## Drop mh_ from the beginning of the variable name
    b.columns=[col.split('su_')[-1].split('_STD_CAT_RESULT')[0].capitalize() for col in b.columns]
    b=b.replace({'Y':'yes','N':'no'},regex=True)
    
    string_dict=b.to_dict(orient='index')
      
    return string_dict

###===================
def return_race_dict():
    data=load_merged_data_of_lab_vars()
    race_df=data.drop_duplicates(subset=['USUBJID'])[['USUBJID','STUDYID','RACE']]
    race_df.loc[race_df['STUDYID']=='TB-1022','RACE']='BLACK'
    race_dict=race_df.set_index('USUBJID')['RACE'].to_dict()

    return race_dict



##=========================================
## 
def backward_fill_and_extract_vars_at_baseline(X_subset,columns_to_drop=None):
    from tqdm import tqdm

    X_subset=X_subset.drop(columns=X_subset.columns[X_subset.columns.str.contains('dr_reg|drugs_cumul|cumul_toxgrade|CULTURE_STATUS')])

    if columns_to_drop is not None:
        ## Extract columns selected for input 
        columns_to_drop_=list(set(X_subset.columns.tolist())&set(columns_to_drop))
        final_cols=X_subset.drop(columns=columns_to_drop_).columns.tolist()
        final_cols=[col for col in final_cols if col !='DAY']
        #print('final_cols',final_cols)
        
        ## Drop visits (==rows), where there are missing data in the selected variable columns
        a=X_subset[['DAY']+final_cols].dropna(how='any',subset=final_cols,axis=0)
        a = a.loc[:,~a.columns.duplicated()].copy()

    if columns_to_drop is None:
        final_cols=[col for col in X_subset.columns.tolist() if col !='DAY']


        ## For each final variable, check the number of  patiens who have only missing values in the first month.
        #. If a variable has a high relative missing rate in the first month across the patients (>0.1, i.e. missing for more than 10 % of patients),
        #. don't backfill that variable 
        num_of_pats_with_missing_vars_in_first_month=X_subset[['DAY']+final_cols].sort_values('DAY').groupby('USUBJID').apply(lambda x:x.loc[x['DAY']<31,:].isna().all()).sum().sort_values()
        num_of_pats_with_missing_vars_in_first_month_=num_of_pats_with_missing_vars_in_first_month/X_subset['USUBJID'].unique().shape[0]
        vars_to_backfill = num_of_pats_with_missing_vars_in_first_month_[num_of_pats_with_missing_vars_in_first_month_<=0.05].index.tolist()
        vars_not_to_backfill = num_of_pats_with_missing_vars_in_first_month_[num_of_pats_with_missing_vars_in_first_month_>0.0].index.tolist()
    
    
        #print('num_of_pats_with_missing_vars_in_first_month_',num_of_pats_with_missing_vars_in_first_month_[num_of_pats_with_missing_vars_in_first_month_>0])
    
        ## Drop variables not to backfill from the final cols
        final_cols_=[fin_col for fin_col in final_cols if fin_col in vars_to_backfill]
        #final_cols_=final_cols
        #print('final_cols_',final_cols_)
        
        ## Drop visits (==rows), where there are missing data in the selected variable columns
        a=X_subset[['DAY']+final_cols_].dropna(how='any',subset=final_cols_,axis=0)
        
        #a=X_subset.copy()
        a = a.loc[:,~a.columns.duplicated()].copy()
        
    #print(a.columns)
    #print(a.head())
    
    ## Extract the first day of study, where the patient has no missing information in the input variables
    ## ==> this way we can check which was the first visit where all of the selected variables have non-missing measurements 
    #. ==> To impute missing variables at earlier visits, backward fill the missing variable's first valid value
    #. ==> The upper limit where a backward fill is acceptable is 31 days.
    first_complete_day_df=a.sort_values(by=['DAY']).groupby('USUBJID').apply(lambda x: x.loc[x.index[0],:])
    
    pats_with_miss_vars=first_complete_day_df[(first_complete_day_df['DAY']>-30) &((first_complete_day_df['DAY']<31))].index
    
    X_subset_=X_subset.copy()


    ## Loop over patients who have missing data in their early visits, and backward fill missing data
    #for pat in tqdm(pats_with_miss_vars[:]):
    for pat in pats_with_miss_vars[:]:
        pat_df=X_subset[X_subset['USUBJID']==pat]
    
        ## Get columns which have NaNs in the final columns containing input variables
        nan_cols_bool=pat_df.loc[:,['DAY']+final_cols].sort_values('DAY').isna().any(axis=0)
        nan_cols=nan_cols_bool[nan_cols_bool.values].index.tolist()#+ ['dr_reg_study_drugs_cumul']
    
        ## backward fill those columns with the first observed value + insert backward filled data into the X_subset dataframe
        #interp_df=pat_df[nan_cols].interpolate('bfill')
        interp_df=pat_df[nan_cols].bfill()
        X_subset_.loc[interp_df.index,nan_cols] = interp_df.values


    #print('X_subset_ head',X_subset_[['DAY']+final_cols].head())
    #print('final_cols')
    #print(X_subset_['USUBJID'].unique().shape)
    
    ## Get all the visits before DAY 5 of the study and drop those visits, where despite of the backward filling, there are still NaNs
    c=X_subset_[['DAY']+final_cols].sort_values(by=['DAY']).groupby('USUBJID').apply(lambda x: x.loc[(x['DAY']<5),:]).dropna(how='any',subset=final_cols,axis=0)#['USUBJID'].unique().shape

    #print('c head',c.head())
    ## If there are multiple early visits (usually in the week prior to therapy start, and then on the first 1-5 days), take the earliest timepoint as the baseline
    #. + drop drug regimen data columns, as theoretically no drugs have been taken yet
    #  + drop cumulate adverse events columns, as there was no therapy 
    #X_subset_baseline=c.drop(columns=['USUBJID']).groupby('USUBJID',as_index=True).apply(lambda x: x.loc[(x.index[0]),:])
    X_subset_baseline=c.drop(columns=['USUBJID']).groupby('USUBJID',as_index=True).apply(lambda x: x.loc[(x.index[0]),:])
    X_subset_baseline=X_subset_baseline.drop(columns=X_subset_baseline.columns[X_subset_baseline.columns.str.contains('dr_reg|drugs_cumul|cumul_toxgrade|CULTURE_STATUS')])
    #X_subset_baseline['index']=np.nan
    print('cols after backfill',print(X_subset_baseline.reset_index().columns))
    
    return X_subset_baseline.reset_index()#.drop(columns='')

### =================
def prepare_baseline_data(key):

    fn='../data/'+key+'_preproc_data_with_imp.csv.gz'
    data_baseline=pd.read_csv(fn,index_col=0)
    data_baseline=data_baseline.rename(columns=lambda x: x.replace('<', 'lower than'))
    data_baseline=data_baseline.rename(columns=lambda x: x.replace('>', 'higher than'))
    data_baseline=data_baseline[data_baseline['DAY']>-100]


    if 'RACE' not in data_baseline.columns:
        data_baseline['RACE']=data_baseline['USUBJID'].map(race_dict)
        #X = pd.concat([X.drop(columns=['RACE']),pd.get_dummies(X['RACE'],dtype=int,prefix='RACE')],axis=1)

    data_baseline['vs_BMI_STD_NUM_RESULT'] = (data_baseline['vs_Weight_STD_NUM_RESULT']/(data_baseline['vs_Height_STD_NUM_RESULT']/100)**2).values
    
    columns_to_drop=['ARM','STUDYID']
    temp_cols_to_drop=['ae','mh','cm','ce'][:-1]
    
    ## Drop all columns that contain only zeroes
    data_baseline=data_baseline.loc[:, (data_baseline!= 0).any(axis=0)]
    
    ## Define baseline columns to keep 
    temp_col_threshold=0.3 # 0.3 
    temporal_data_names=['re','ae','cm','su','mh'][:-1] 
    temp_cols_to_keep=['dr_reg_study_drugs_cumul','vs_Height_STD_NUM_RESULT',
                      'vs_BMI_STD_NUM_RESULT','RACE'] 
    
    for temp_data_name in temporal_data_names:
        cols_to_keep=select_temporal_cols_with_suff_pat_data(temp_data_name,data_baseline,temp_col_threshold)
        temp_cols_to_keep.extend(cols_to_keep)
    
    columns_to_drop_=columns_to_drop + data_baseline.columns[data_baseline.columns.str.startswith(tuple(temp_cols_to_drop))].tolist()
    
    ## If colunm name is in the temp_cols_to_keep list, don't drop it
    columns_to_drop_=[coln for coln in columns_to_drop_ if coln not in temp_cols_to_keep]
    #data_baseline=data_baseline.drop(columns=columns_to_drop_)

    return data_baseline,columns_to_drop_

### =================
# Some of the columns (containing the std units) were droped during imputation. Extract these columns from the concatenated dataset
#. + Convert the 1/0 back to positive and negative where they were converted druing imputation
def add_missing_ds_type_var_colnames(ds_type_var_colnames,data_conc_period,data_conc_):
    
    non_verlap_cols=[coln for coln in ds_type_var_colnames if coln not in data_conc_period.columns]
    non_verlap_cols = non_verlap_cols + ['DAY','USUBJID']
    data_conc_period = pd.merge(data_conc_period,data_conc_.loc[:,non_verlap_cols],how='left',on=['DAY','USUBJID'])

    ## COnvert back to positive/neagtive
    mb_cols_to_ordinal=['mb_ZN-smear_STD_RESULT', 'mb_MGIT_STD_RESULT',
                        'mb_LJ-culture_CULTURE_STATUS','mb_MGIT_CULTURE_STATUS',
                       'mb_HAIN-test_STD_RESULT', 'mb_MTB-complex_STD_RESULT',
                       'mb_Auramine-smear_STD_RESULT', 'mb_LJ-culture_STD_RESULT',
                       'mb_AccuProbe_STD_RESULT', 'mb_MPT64-Antigen-Test_STD_RESULT',
                       'mb_RT-PCR_STD_RESULT']
        
    for mb_col in mb_cols_to_ordinal:
        if mb_col in data_conc_period.columns:
            data_conc_period.loc[:,mb_col]=data_conc_period.loc[:,mb_col].replace({1:'positive',0:'negative'},regex=True)

    return data_conc_period



## Load vars

In [4]:
for key in [*parameters_for_analysis][:1]:
    
    fn='../data/'+key+'_all_data_concat.csv.gz'
    data_conc=pd.read_csv(fn,low_memory=False,index_col=0)
    data_conc=data_conc[data_conc['DAY']>-100]
    
    all_vars_dict,all_vars=get_all_var_names()
    vars_left_for_anal=list(set(all_vars) - set(data_conc.columns[data_conc.apply(lambda x: all(x.isna()))]))

## Run conversion

In [18]:
data_baseline.loc[:,data_baseline.columns.str.startswith('mb_')].columns

Index(['mb_ZN-smear_STD_RESULT', 'mb_ZN-smear_STD_CAT_ORDINAL_RESULT',
       'mb_LJ-culture_STD_RESULT', 'mb_LJ-culture_CULTURE_STATUS'],
      dtype='object')

In [8]:
import json



## Dictionary containing the respective function for each dataset type
conv_funct_dict={'mb':convert_mb_vars,
                 'dm':convert_dm_vars,
                 'vs':convert_vs_vars,
                 're':convert_re_vars,
                 'lb':convert_lb_vars,
                 'dr_reg':convert_dr_reg_vars,
                 'mh':convert_mh_vars,
                 'ce':convert_ce_vars,
                 'cmdos':convert_cmdos_vars,
                 #'cmind':convert_cmind_vars,
                 'ms':convert_ms_vars,
                 'ae':convert_ae_vars,
                 'su':convert_su_vars}





## Last days of periods to use as training data.
# i.e period_end_day=31 ==> use only data of patient coming from the first 30 days
period_end_days=[31,62,93,125,160,'all']
period_end_days=['baseline',31,62,93,125,160,'all']

## Data inclusion type: all data in given perios is converted, or inly the data from the last day in time period
#data_inclusion_types=['all_days','baseline_vars','baseline_last_day']
data_inclusion_types=['baseline_last_day','baseline_vars','all_days']


## Dataset types: - static vars, which don't change over time
#                 -  dynamically measured variables: measured repeatedly over therapy period
var_type_dict={'static_vars':['dm','re','vs','mh','ce','su'],
                 'dynamic_vars':['mb','lb','dr_reg','cmdos','ms']}

## Dataset-types to consider (change 'cm' to 'cmdos')
ds_types_list=[*all_vars_dict]

ds_types_list=['cmdos' if ds=='cm' else ds for ds in ds_types_list]
all_vars_dict['cmdos']=[ds.replace('cm_','cmdos_').lower() for ds in all_vars_dict['cm']]
all_vars=[ds.replace('cm_','cmdos_').lower() if ds.startswith('cm_') else ds for ds in all_vars]

#all_vars_dict['ms']=[ds.lower() for ds in all_vars_dict['ms']]
#all_vars=[ds.lower() for ds in all_vars if ds.startswith('ms_')]

## Return a dictionary containing the race of the patients
race_dict=return_race_dict()



## LOOP THROUGH THE DIFFERENT DATASETS AND ALL THE DIFFERENT DATA LAYERS, AND CONVERT TABULAR DATA INTO DICTIONARY OF STRINGS
for key in [*parameters_for_analysis][:1]:

    for data_inclusion_type in data_inclusion_types[-1:]:
        print(f'\n======= {key} - {data_inclusion_type}========\n')
    
        #if data_inclusion_type=='all_days':
        fn='../data/'+key+'_all_data_concat.csv.gz'
        data_conc=pd.read_csv(fn,low_memory=False,index_col=0)
        data_conc=data_conc[data_conc['DAY']>-100]
        data_conc['RACE']=data_conc['USUBJID'].map(race_dict)

        #if data_inclusion_type in ['baseline_vars','baseline_last_day']:
        data_baseline,columns_to_drop_ = prepare_baseline_data(key)
        data_baseline=data_baseline.drop(columns=columns_to_drop_)

        ## Replace extreme Hyperkalemia values with mean (probably false measurements)
        mean_K=data_baseline.loc[data_baseline['lb_Blood Potassium_STD_NUM_RESULT']<=12,'lb_Blood Potassium_STD_NUM_RESULT'].mean()
        data_baseline.loc[data_baseline['lb_Blood Potassium_STD_NUM_RESULT']>12,'lb_Blood Potassium_STD_NUM_RESULT']=mean_K
        data_conc.loc[data_conc['lb_Blood Potassium_STD_NUM_RESULT']>12,'lb_Blood Potassium_STD_NUM_RESULT']=mean_K
            

        for end_day in period_end_days[1:2]:
    
            print('\n=======',data_inclusion_type,end_day,'days========\n')

            if end_day=='baseline':                 
                #if data_inclusion_type=='all_days':
                data_conc_=data_conc[data_conc['DAY']<=31]#.dropna(how='all',axis=1)
                
                #if data_inclusion_type in ['baseline_vars','baseline_last_day']:
                data_baseline_=data_baseline[data_baseline['DAY']<=31]
            
            if isinstance(end_day,int):
                #data_conc_=data_conc[data_conc['DAY']<=end_day]#.dropna(how='all',axis=1)
                #data_baseline_=data_baseline[data_baseline['DAY']<=end_day]

                ## Select last visit in period cutoff based on threshold before cutoff & after cutoff
                data_conc_ = select_visits_with_dual_thresholds(
                                                df=data_conc,
                                                time_col='DAY',
                                                patient_col='USUBJID',
                                                cutoff_day=end_day,
                                                before_threshold=20,
                                                after_threshold=10)

                ## Select last visit in period cutoff based on threshold before cutoff & after cutoff
                data_baseline_ = select_visits_with_dual_thresholds(
                                                df=data_baseline,
                                                time_col='DAY',
                                                patient_col='USUBJID',
                                                cutoff_day=end_day,
                                                before_threshold=20,
                                                after_threshold=10)

            
            
            if end_day=='all': 
                data_conc_=data_conc.copy()#.dropna(how='all',axis=1)
                data_baseline_=data_baseline.copy()

            #print(data_conc_['DAY'].max())
            #print(data_baseline_['DAY'].max())
    
            ## Extract variables that have at least one entry
            if data_inclusion_type in ['baseline_vars','baseline_last_day']:
                #vars_left_for_anal=list(set(all_vars) - set(data_baseline.columns[data_baseline.apply(lambda x: all(x.isna()))]))
                vars_left_for_anal = [var_ for var_ in all_vars for coln in data_baseline.columns if coln.startswith(var_)]

            if data_inclusion_type not in ['baseline_vars','baseline_last_day']:
                vars_left_for_anal=list(set(all_vars) - set(data_conc.columns[data_conc.apply(lambda x: all(x.isna()))]))
    
            
            ## Create list to collect the different dataframes of the dataypes, to concatenate at the end
            #. => This will be used to correlate with LLM models predictions, to help understand the individual variables impact on prediction
            df_list=[]
            
            
            
            for ds_type in ['ae']: #[*conv_funct_dict][:]:    #ds_types_list[13:14]:# + ['dr_reg']:
                print(ds_type)
                
                if ds_type=='dr_reg':
                    ds_type_var_colnames=[coln for coln in data_conc_.columns if coln.startswith(ds_type)]
                    #ds_type_vars=[x.split('_cumulative_dose')[0] for x in ds_type_var_colnames]
                    ds_type_vars=[x.split('_cumulative_dose')[0] for x in ds_type_var_colnames if x!='dr_reg_study_drugs_cumul'] + ['dr_reg_study_drugs_cumul']
                elif ds_type=='mh':
                    ds_type_var_colnames = data_conc.columns[data_conc.columns.str.startswith('mh')].tolist()
                else:        
                    ds_type_vars=list(set(all_vars_dict[ds_type])&set(vars_left_for_anal))
                    ds_type_var_colnames=[coln for coln in data_conc_.columns for ds_type_var in ds_type_vars if coln.startswith(ds_type_var)]          
            
                #print(data_inclusion_type)

                if len(ds_type_var_colnames)==0 and ds_type!='mh':
                    print(f'No {ds_type} variables! Skipping to next dataset type!')
                    continue

                if end_day=='baseline' and ds_type in ['dr_reg','ae','cmdos']:
                     print(f'Skip {ds_type} variables at baseline!')
                     continue
                    

                ## For the experimental setups with time series data, backward fill if baseline time point is considered, else take the time series data
                if data_inclusion_type=='all_days': 
                    if end_day=='baseline': 
                        data_conc_period = data_conc_.loc[:,['USUBJID','DAY'] + ds_type_var_colnames].sort_values(['USUBJID','DAY']).groupby(['USUBJID']).apply(lambda x: x.bfill())
                        data_conc_period=data_conc_period[(data_conc_period['DAY']>-30)&(data_conc_period['DAY']<5)]
                        data_conc_period = data_conc_period.drop(columns=['USUBJID']).groupby('USUBJID',as_index=False).apply(lambda x: x.ffill().bfill())
                        data_conc_period=data_conc_period.groupby('USUBJID',as_index=True).apply(lambda x: x.loc[(x.index[0]),:]).reset_index()
                        data_conc_period=data_conc_period.loc[:,~data_conc_period.columns.str.contains('CULTURE_STATUS')]
                        ds_type_var_colnames=[c for c in ds_type_var_colnames if 'CULTURE_STATUS' not in c]

                    if end_day!='baseline': 
                        data_conc_period=data_conc_.copy()

                if data_inclusion_type=='baseline_vars':                    
                   
                    if end_day=='baseline': 
                        #data_conc_period = backward_fill_and_extract_vars_at_baseline(data_baseline_,columns_to_drop_)
                        if ds_type=='dm':
                            data_conc_period=data_baseline_.copy()
                            data_conc_period['RACE']=data_conc_period['USUBJID'].map(race_dict)
                            data_conc_period=data_conc_period.loc[:,['USUBJID','DAY'] + ds_type_var_colnames]  
                            data_conc_period = data_conc_period.drop_duplicates('USUBJID')
                            

                        if ds_type!='dm':                            
                            data_conc_period=data_baseline_.copy()
                            
                            ## Drop columns with all NaNs, as this interferes with the backward fill
                            cols_with_all_nans=data_conc_period.columns[data_conc_period.isna().all().values].tolist()
                            data_conc_period=data_conc_period.dropna(how='all',axis=1)

                            ## Some dr_reg columns have only zeros and NaNs (these drugs werent taken by any of the patients)
                            ## Drop these columns as they only mess up the backward fill
                            cols_wo_information=((data_conc_period==0)|(data_conc_period.isna())).all()
                            dr_reg_cols_to_drop=cols_wo_information[cols_wo_information].index.tolist()
                            data_conc_period=data_conc_period.drop(columns=dr_reg_cols_to_drop)
                            
                            print('starting backward fill')
                            
                            data_conc_period = backward_fill_and_extract_vars_at_baseline(data_conc_period)#,cols_to_drop__)
                            

                            ## During imputation some of the unnecessary columns were dropped (i.e. standard units, etc..), because these unnecassary for the baseline raw models
                            ## ==> load the concatenated dataset as it contains them, and concatenate these columns, as they are necessary for the LLM embedding
                            data_conc_period = add_missing_ds_type_var_colnames(ds_type_var_colnames,data_conc_period,data_conc_)
                            data_conc_period.loc[:,data_conc_period.columns.str.endswith('STD_NUM_UNIT')]=data_conc_period.loc[:,data_conc_period.columns.str.endswith('STD_NUM_UNIT')].bfill()
                            #data_conc_period[cols_with_all_nans]=np.nan
                            data_conc_period=data_conc_period.loc[:,~data_conc_period.columns.str.contains('CULTURE_STATUS')]
                            ds_type_var_colnames=[c for c in ds_type_var_colnames if 'CULTURE_STATUS' not in c]

                    if end_day!='baseline':       
                        data_conc_period=data_baseline_.copy()
                        data_conc_period['RACE']=data_conc_period['USUBJID'].map(race_dict)

                    ## During imputation some of the unnecessary columns were dropped (i.e. stanard units, etc..), because these unnecassary for the baseline raw models
                    ## ==> load the concatenated dataset as it contains them, and concatenate these columns, as they are necessary for the LLM embedding
                    data_conc_period = add_missing_ds_type_var_colnames(ds_type_var_colnames,data_conc_period,data_conc_)


                ## For the experimental setup of last day in period:
                # Some of the columns (containing the std units) were droped during imputation. Extract these columns from the concatenated dataset
                #. + Convert the 1/0 back to positive and negative where they were converted druing imputation
                if data_inclusion_type=='baseline_last_day':
                    
                    if end_day!='baseline':
                        ## Some patients' demographic data got lost during imputation ==> load the concatenated dataset, as it contains them
                        if ds_type=='dm':
                            data_conc_period=data_conc_.copy()
                        
                        if ds_type!='dm':
                            data_conc_period=data_baseline_.copy()

                        ## During imputation some of the unnecessary columns were dropped (i.e. stanard units, etc..), because these unnecassary for the baseline raw models
                        ## ==> load the concatenated dataset as it contains them, and concatenate these columns, as they are necessary for the LLM embedding
                        data_conc_period = add_missing_ds_type_var_colnames(ds_type_var_colnames,data_conc_period,data_conc_)
    
                        ## Extract last day                
                        
                        ## In some cases the last day information is imputed 
                        ## ==> in these cases, the matching columns containing measurement units are NaN, which are extracted
                        #.     and merged from the data_conc dataframe 
                        #. ==> forward fill, so columns which contain string (like measurement units) and belong to imputed datapoints, are captured
                        data_conc_period = data_conc_period.groupby('USUBJID',as_index=False).apply(lambda x: x.sort_values(by=['DAY'],ascending=True).ffill().loc[x['DAY']==x['DAY'].max(),:])

                    if end_day=='baseline':
                        if ds_type=='dm':
                            data_conc_period=data_baseline_.copy()
                            data_conc_period['RACE']=data_conc_period['USUBJID'].map(race_dict)
                            data_conc_period=data_conc_period.loc[:,['USUBJID','DAY'] + ds_type_var_colnames]  
                            data_conc_period = data_conc_period.groupby('USUBJID',as_index=False).apply(lambda x: x.ffill().bfill()).drop_duplicates('USUBJID').reset_index().drop(columns=['level_0','level_1'])


                        if ds_type!='dm':                            
                            data_conc_period=data_baseline_.copy()
                            
                            ## Drop columns with all NaNs, as this interferes with the backward fill
                            cols_with_all_nans=data_conc_period.columns[data_conc_period.isna().all().values].tolist()
                            data_conc_period=data_conc_period.dropna(how='all',axis=1)

                            ## Some dr_reg columns have only zeros and NaNs (these drugs werent taken by any of the patients)
                            ## Drop these columns as they only mess up the backward fill
                            cols_wo_information=((data_conc_period==0)|(data_conc_period.isna())).all()
                            dr_reg_cols_to_drop=cols_wo_information[cols_wo_information].index.tolist()
                            data_conc_period=data_conc_period.drop(columns=dr_reg_cols_to_drop)

                            print('starting backward fill')
                            
                            data_conc_period = backward_fill_and_extract_vars_at_baseline(data_conc_period)#,cols_to_drop__)
                            

                            ## During imputation some of the unnecessary columns were dropped (i.e. stanard units, etc..), because these unnecassary for the baseline raw models
                            ## ==> load the concatenated dataset as it contains them, and concatenate these columns, as they are necessary for the LLM embedding
                            data_conc_period = add_missing_ds_type_var_colnames(ds_type_var_colnames,data_conc_period,data_conc_)
                            data_conc_period.loc[:,data_conc_period.columns.str.endswith('STD_NUM_UNIT')]=data_conc_period.loc[:,data_conc_period.columns.str.endswith('STD_NUM_UNIT')].bfill()
                            #data_conc_period[cols_with_all_nans]=np.nan
                            data_conc_period=data_conc_period.loc[:,~data_conc_period.columns.str.contains('CULTURE_STATUS')]
                            ds_type_var_colnames=[c for c in ds_type_var_colnames if 'CULTURE_STATUS' not in c]
                            #print('adding some vars after backfill',data_conc_period.columns)

                          
                
                df=data_conc_period.loc[:,['USUBJID','DAY'] + ds_type_var_colnames]  
                print('df.shape',df.shape)

                ## CREATE DATAFRAMW WITH NUMERICAL VALUES TO CORRELATE WITH MODEL PREDICTION PROBABILITIES LATER
                ## Drop non-numerical columns containing measurement units, names of the variable, and scaled num values
                #. + drop all columns with NaNs + convert mb tests back to numerical ('positive, negative')
                df_for_corr= df.loc[:,(~df.columns.str.endswith('_UNIT'))\
                                &(~df.columns.str.contains('_SCALED|NUM_TEST|CAT_TEST'))].dropna(how='all',axis=1).replace({'positive':1,'negative':0})

                ## Drop AE columns that have entries in a low number of patients
                if ds_type=='ae':
                    temp_col_threshold=0.005 # 0.3 
                    cols_to_keep=select_temporal_cols_with_suff_pat_data('ae',data_conc,temp_col_threshold) 
                    ae_cols_to_drop=[col for col in df_for_corr.columns[df_for_corr.columns.str.startswith('ae')] if col not in cols_to_keep]
                    df_for_corr = df_for_corr.drop(columns=ae_cols_to_drop)
                
                df_list.append(df_for_corr)

                
                ## Convert dataframe to sentences, which will be stored in a dictionary
                conv_func=conv_funct_dict[ds_type] 
                sentence_dict=conv_func(df,ds_type_vars,ds_type)

                
                # Specify the file path where you want to save the JSON data
                fn=f'../data/{key}_{ds_type}_{end_day}_days_{data_inclusion_type}_string_converted_dict.json'
    
                # Save the dictionary as JSON
                #with open(fn, 'w') as json_file:
                #    json.dump(sentence_dict, json_file)
                
            ## For baseline timepoint for the baselien_last_day inclusion type, set the USUBJID as index, otherwise when concatenating some USUBJIDs,
            #. that are missing in some ds_types (i.e.ce, vs,lb), will also be missing in the dataframe
            if (data_inclusion_type=='baseline_vars' and end_day=='baseline') or data_inclusion_type=='baseline_last_day':
                df_list=[d.set_index('USUBJID') for d in df_list]
            
            df_concat=pd.concat(df_list,axis=1)   
            df_concat = df_concat.loc[:,~df_concat.columns.duplicated(keep='first')]
            print('df_concat.shape',df_concat.shape)
            fn=f'../data/{key}_{end_day}_days_{data_inclusion_type}_data_for_embedding.csv.gz'
            df_concat.to_csv(fn)
                


======= tb21_22_2984_pats_22_vars_result_at_end_of_treatment - all_days========


======= all_days 31 days========

Excluded 0 out of 2984 patients (0.0%)

Excluded 0 out of 2984 patients (0.0%)

ae
df.shape (17075, 1528)
df_concat.shape (17075, 50)


In [9]:
sentence_dict['TB-1021/1084727']

'Dyspnea: Week 1-4, severity 2; Joint pain: Week 1-2, severity 2'

In [81]:
data_conc.loc[~data_conc['mb_AccuProbe_STD_RESULT'].isna(),'mb_AccuProbe_STD_RESULT']#.dropna()

57       positive
82       positive
95       positive
186      positive
200      positive
           ...   
19506    positive
19516    positive
19517    positive
19518    positive
19519    positive
Name: mb_AccuProbe_STD_RESULT, Length: 1789, dtype: object

In [72]:
df[~df['mb_AccuProbe_STD_RESULT'].isna()].dropna(how='all',axis=0)

USUBJID    DAY mb_ZN-smear_STD_RESULT  \
57     TB-1021/1086597    1.0               positive   
82     TB-1021/1089792  183.0               negative   
95     TB-1021/1090240  155.0               negative   
186    TB-1021/1104035    1.0               positive   
200    TB-1021/1104271    1.0               positive   
...                ...    ...                    ...   
19506  TB-1021/2811814    1.0               positive   
19516  TB-1021/2811814  120.0               negative   
19517  TB-1021/2811814  162.0               positive   
19518  TB-1021/2811814  176.0               positive   
19519  TB-1021/2811814  183.0               positive   

      mb_ZN-smear_STD_CAT_RESULT  mb_ZN-smear_CULTURE_STATUS  \
57                            2+                         NaN   
82                      NEGATIVE                         NaN   
95                      NEGATIVE                         NaN   
186                           4+                         NaN   
200                           4+                         NaN   
...                          ...                         ...   
19506                         3+                         NaN   
19516                   NEGATIVE                         NaN   
19517                         1+                         NaN   
19518                         3+                         NaN   
19519                         3+                         NaN   

       mb_ZN-smear_STD_NUM_UNIT  mb_ZN-smear_STD_CAT_ORDINAL_RESULT  \
57                          NaN                                 2.0   
82                          NaN                                 0.0   
95                          NaN                                 0.0   
186                         NaN                                 4.0   
200                         NaN                                 4.0   
...                         ...                                 ...   
19506                       NaN                                 3.0   
19516                       NaN                                 0.0   
19517                       NaN                                 1.0   
19518                       NaN                                 3.0   
19519                       NaN                                 3.0   

       mb_ZN-smear_STD_NUM_RESULT mb_MGIT_STD_RESULT  mb_MGIT_STD_CAT_RESULT  \
57                            NaN           positive                     NaN   
82                            NaN           positive                     NaN   
95                            NaN           positive                     NaN   
186                           NaN           positive                     NaN   
200                           NaN           positive                     NaN   
...                           ...                ...                     ...   
19506                         NaN           positive                     NaN   
19516                         NaN           positive                     NaN   
19517                         NaN           positive                     NaN   
19518                         NaN           positive                     NaN   
19519                         NaN           positive                     NaN   

       ... mb_MPT64-Antigen-Test_CULTURE_STATUS  \
57     ...                                  NaN   
82     ...                                  NaN   
95     ...                                  NaN   
186    ...                                  NaN   
200    ...                                  NaN   
...    ...                                  ...   
19506  ...                                  NaN   
19516  ...                                  NaN   
19517  ...                                  NaN   
19518  ...                                  NaN   
19519  ...                                  NaN   

      mb_MPT64-Antigen-Test_STD_NUM_UNIT  \
57                                   NaN   
82                                   NaN   
95            

In [69]:
df.columns

Index(['USUBJID', 'DAY', 'mb_ZN-smear_STD_RESULT',
       'mb_ZN-smear_STD_CAT_RESULT', 'mb_ZN-smear_CULTURE_STATUS',
       'mb_ZN-smear_STD_NUM_UNIT', 'mb_ZN-smear_STD_CAT_ORDINAL_RESULT',
       'mb_ZN-smear_STD_NUM_RESULT', 'mb_MGIT_STD_RESULT',
       'mb_MGIT_STD_CAT_RESULT', 'mb_MGIT_CULTURE_STATUS',
       'mb_MGIT_STD_NUM_UNIT', 'mb_MGIT_STD_CAT_ORDINAL_RESULT',
       'mb_MGIT_STD_NUM_RESULT', 'mb_MTB-complex_STD_RESULT',
       'mb_MTB-complex_STD_CAT_RESULT', 'mb_MTB-complex_CULTURE_STATUS',
       'mb_MTB-complex_STD_NUM_UNIT', 'mb_MTB-complex_STD_CAT_ORDINAL_RESULT',
       'mb_MTB-complex_STD_NUM_RESULT', 'mb_HAIN-test_STD_RESULT',
       'mb_HAIN-test_STD_CAT_RESULT', 'mb_HAIN-test_CULTURE_STATUS',
       'mb_HAIN-test_STD_NUM_UNIT', 'mb_HAIN-test_STD_CAT_ORDINAL_RESULT',
       'mb_HAIN-test_STD_NUM_RESULT', 'mb_Auramine-smear_STD_RESULT',
       'mb_Auramine-smear_STD_CAT_RESULT', 'mb_Auramine-smear_CULTURE_STATUS',
       'mb_Auramine-smear_STD_NUM_UNIT',
       'mb_

In [11]:
df

USUBJID   DAY  dr_reg_pyrazinamide_cumulative_dose  \
0      TB-1021/1084727  -1.0                                  NaN   
1      TB-1021/1084727   1.0                               1000.0   
2      TB-1021/1084727   8.0                               8000.0   
3      TB-1021/1084727  15.0                              15000.0   
4      TB-1021/1084727  22.0                              22000.0   
...                ...   ...                                  ...   
37611  TB-1021/2466232   1.0                               1000.0   
37612  TB-1021/1041599   1.0                               1000.0   
37613  TB-1021/1893470   1.0                               1500.0   
37614  TB-1021/2529741   1.0                               1500.0   
37615  TB-1021/2389364   1.0                               1000.0   

       dr_reg_rifapentine_cumulative_dose  dr_reg_ethambutol_cumulative_dose  \
0                                     NaN                                NaN   
1                                     0.0                              600.0   
2                                     0.0                             4800.0   
3                                     0.0                             9000.0   
4                                     0.0                            13200.0   
...                                   ...                                ...   
37611                                 0.0                              800.0   
37612                                 0.0                                0.0   
37613                                 0.0                                0.0   
37614                                 0.0                                0.0   
37615                                 0.0                                0.0   

       dr_reg_rifampicin_cumulative_dose  dr_reg_moxifloxacin_cumulative_dose  \
0                                    NaN                                  NaN   
1                                  450.0                                400.0   
2                                 3600.0                               3200.0   
3                                 6750.0                               6000.0   
4                                 9900.0                               8800.0   
...                                  ...                                  ...   
37611                              600.0                                400.0   
37612                              450.0                                400.0   
37613                              600.0                                400.0   
37614                              600.0                                400.0   
37615                              450.0                                400.0   

       dr_reg_placebo_cumulative_dose  dr_reg_isoniazid_cumulative_dose  \
0                                 NaN                               NaN   
1                                 1.0                               0.0   
2                                 8.0                               0.0   
3                                15.0                               0.0   
4                                22.0                               0.0   
...                               ...                               ...   
37611                             1.0                               0.0   
37612                             1.0                             300.0   
37613                             1.0                             300.0   
37614                             1.0                             300.0   
37615                             1.0                             300.0   

       dr_reg_linezolid_cumulative_dose  dr_reg_bedaquiline_cumulative_dose  \
0                                   NaN                                 NaN   
1                                   0.0                                 0.0   
2                                   0.0                                 0.0   
3                                  

In [9]:
sentence_dict['TB-1021/1041375']

{'Day 118': 'Haemoptysis: No; Sweat: No; Fever: No; Cough: Yes; Chest pain: Yes'}

In [23]:
df[df['USUBJID']=='TB-1021/1041375']

USUBJID    DAY  ce_HAEMOPTYSIS_STD_CAT_ORDINAL_RESULT  \
95 1193  TB-1021/1041375  118.0                                    0.0   

         ce_HAEMOPTYSIS_STD_CETOXGR ce_HAEMOPTYSIS_STD_CAT_RESULT  \
95 1193                         1.0                             N   

         ce_SWEAT_STD_CAT_ORDINAL_RESULT  ce_SWEAT_STD_CETOXGR  \
95 1193                              0.0                   NaN   

        ce_SWEAT_STD_CAT_RESULT  ce_FEVER_STD_CAT_ORDINAL_RESULT  \
95 1193                       N                              0.0   

         ce_FEVER_STD_CETOXGR ce_FEVER_STD_CAT_RESULT  \
95 1193                   NaN                       N   

         ce_COUGH_STD_CAT_ORDINAL_RESULT  ce_COUGH_STD_CETOXGR  \
95 1193                              1.0                   3.0   

        ce_COUGH_STD_CAT_RESULT  ce_CHEST PAIN_STD_CAT_ORDINAL_RESULT  \
95 1193                       Y                                   1.0   

         ce_CHEST PAIN_STD_CETOXGR ce_CHEST PAIN_STD_CAT_RESULT  
95 1193                        2.0                            Y

In [15]:
d_['TB-1021/1041375']

{'Day 118': 'Cough: Yes; Chest pain: Yes'}

# Create input string for each patient & for each time period

In [10]:
def create_input_string_for_patients(pat_ids,period_end_day,data_inclusion_type,key):
    import random
    
    input_dict_all_ds_types_merged={}
    input_dict_ds_types_seperate={}
    
    print(period_end_day)
    
    #all_var_types=list([*all_vars_dict][:6]) + ['dr_reg']

    all_var_types=['mb',
                     'dm',
                     'vs',
                     're',
                     'lb',
                     'dr_reg',
                     'mh',
                     'ce',
                     'cmdos',
                     #'cmind':convert_cmind_vars,
                     'ms',
                     'ae',
                     'su']
    
    
    
    
    
    if period_end_day=='baseline':
        all_var_types=['mb',
                     'dm',
                     'vs',
                     're',
                     'lb',
                     #'dr_reg',
                     'mh',
                     'ce',
                     'cmdos',
                     #'cmind':convert_cmind_vars,
                     'ms',
                     #'ae',
                     'su']

    '''
    DEPRECATED
    ## If data_inclusion_type=='baseline_vars_ext', use all vraiables available in a given dataset type, but select which dataset types to embed
    if data_inclusion_type=='baseline_vars_ext':
        all_var_types=[  'mb',
                         'dm',
                         'vs',
                         're',
                         'lb',
                         'dr_reg',
                         #'mh',
                         #'ce',
                         #'cmdos',
                         #'cmind':convert_cmind_vars,
                         #'ms',
                         #'ae',
                         #'su'
                          ]
        ## Change data_inclusion_type to all_days, that way all the variables will be loaded in the given dataset type
        data_inclusion_type = 'all_days'
    '''    
        
    #random.Random(4).shuffle(all_var_types)
    
    for pat in list(pat_ids)[:]:
        #print()
        input_dict_ds_types_seperate[pat]={}

        l=[]
           
        #for ds_type in [*all_vars_dict][:6] + ['dr_reg']:
        for ds_type in all_var_types[:]:
            #print(ds_type)

          
                

            # Specify the file path where you want to save the JSON data
            #fn=f'../data/{key}_{ds_type}_string_converted_dict.json'
            #fn=f'../data/{key}_{ds_type}_{period_end_day}_days_string_converted_dict.json'
            key_=parameters_for_analysis[key]['fn']
            fn=f'../data/{key_}_{ds_type}_{period_end_day}_days_{data_inclusion_type}_string_converted_dict.json'
            
            try:
                # Load the JSON data back into a Python dictionary
                with open(fn, 'r') as json_file:
                    ds_type_dict=json.load(json_file)
            except FileNotFoundError:
                continue
            
            
            ## Check if patient is missing any of the dataset types
            #  If drug regimen or lab variables are missing, drop the patient
            if pat not in ds_type_dict.keys():
                
                if ds_type=='dr_reg' or ds_type=='lb':
                    print(f'Dropping {pat} due to missing {ds_type} data!')
                    break
                        
                    
                else:    
                    #print(f'{pat} missing {ds_type} data!')
                    continue

            elif 'wo_dr_reg' in key and ds_type=='dr_reg':
                continue
                
            else:  
                ## Get the initial description of the dataset type (i.e. 'mb':'Microbiological test results')
                ds_type_descr=ds_type_descriptions[ds_type]

                ## Loop over all the results of a given dataset type and concatenate them into one string for the patient
                #  i.e.: Day -2:(Culture growth: Positive; Identification: Positive; Categorical count: Positive); Day 1: ...
                if  ds_type!='ae': #ds_type!='mh
                    ds_type_var_concat='; '.join([f'{key}: {value}' for key,value in ds_type_dict[pat].items()])
                    #print(pat,ds_type_var_concat)

                if ds_type=='ae':
                    ds_type_var_concat = ds_type_dict[pat]
                
                #if ds_type=='mh':
                #    ds_type_var_concat=', '.join([mh_var for mh_var in ds_type_dict[pat]]) #'; '.join([f'{pat}: {ds_type_dict[pat]}'])

                ## Add description of the dataset type in front of the concatenated 
                if ds_type=='dr_reg':
                    ds_type_var_with_descr=f'{ds_type_descr} by patient: [{ds_type_var_concat}]'
                else:
                    ds_type_var_with_descr=f'{ds_type_descr} of patient: [{ds_type_var_concat}]'

                l.append(ds_type_var_with_descr)

                #print(ds_type_var_with_descr)

                input_dict_ds_types_seperate[pat][ds_type]=ds_type_var_with_descr

        #l[0]=prompt
        pat_input_merged='\n'.join(l)

        input_dict_all_ds_types_merged[pat]=pat_input_merged
        
        #print(pat_input_merged)


    ### SAVE INPUT DICTIONARIES    
    # Specify the file path where you want to save the JSON data
    fn=f'../data/{key}_{period_end_day}_days_{data_inclusion_type}_input_dict_all_ds_types_merged.json'

    #print(input_dict_all_ds_types_merged)
    # Save the dictionary as JSON
    with open(fn, 'w') as json_file:
        json.dump(input_dict_all_ds_types_merged, json_file)


    fn=f'../data/{key}_{period_end_day}_days_{data_inclusion_type}_input_dict_ds_types_seperate.json'

    # Save the dictionary as JSON
    with open(fn, 'w') as json_file:
        json.dump(input_dict_ds_types_seperate, json_file)





## LOOP THROUGH THE DIFFERENT DATASETS AND ALL THE DIFFERENT DATA LAYERS, AND CONVERT TABULAR DATA INTO DICTIONARY OF STRINGS
for key in [*parameters_for_analysis][:1]:
    ## Load data
    fn='../data/'+key+'_all_data_concat.csv.gz'
    data_conc=pd.read_csv(fn,low_memory=False,index_col=0,usecols=range(5))
    pat_ids=data_conc['USUBJID'].unique().tolist()



## Define string descriptions to add as a prefix for each ds_type, for LLM to know what the variables describe
ds_type_descriptions={  'dm':'Demographic descriptors',
                        'mb':'Microbiological test results',
                        'vs':'Vital signs',
                        're':'Chest X-ray findings',
                        'lb':'Laboratory test results',
                        'dr_reg':'Cumulative drug doses taken',
                        'mh':'Medical history',
                        'ms':'Microbiological susceptibility',
                        'cmdos':'Cumulative concomitant medication taken',
                        'ce':'Clinical events',
                        'su':'Substance use',
                        'ae':'Adverse events'}



for key in [*parameters_for_analysis][:1]:
    print(key)
    
    ## Last days of periods to use as training data.
    # i.e period_end_day=31 ==> use only data of patient coming from the first 30 days
    period_end_days=['baseline',31,62,93,125,160,'all']
    
    ## Data inclusion type: all data in given perios is converted, or inly the data from the last day in time period
    data_inclusion_types=['all_days','last_day']
    data_inclusion_types=['baseline_last_day','baseline_vars','all_days']#'baseline_vars_ext']
    
    for period_end_day in period_end_days[:1]:
    
        for data_inclusion_type in data_inclusion_types[1:]:
            print(data_inclusion_type)
            create_input_string_for_patients(pat_ids,period_end_day,data_inclusion_type,key)
    
    print('Done!')

 

tb21_22_2984_pats_22_vars_result_at_end_of_treatment
baseline_vars
baseline
Dropping TB-1021/2632780 due to missing lb data!
Dropping TB-1021/2684790 due to missing lb data!
Dropping TB-1022/42072 due to missing lb data!
Dropping TB-1022/42092 due to missing lb data!
Dropping TB-1022/42173 due to missing lb data!
Dropping TB-1022/43213 due to missing lb data!
Dropping TB-1022/52351 due to missing lb data!
Dropping TB-1022/53118 due to missing lb data!
Dropping TB-1022/53136 due to missing lb data!
Dropping TB-1022/53148 due to missing lb data!
Dropping TB-1022/53487 due to missing lb data!
Dropping TB-1022/53506 due to missing lb data!
Dropping TB-1022/21208 due to missing lb data!
Dropping TB-1022/11020 due to missing lb data!
Dropping TB-1022/11021 due to missing lb data!
Dropping TB-1022/11039 due to missing lb data!
Dropping TB-1022/11141 due to missing lb data!
Dropping TB-1022/11150 due to missing lb data!
Dropping TB-1022/11151 due to missing lb data!
Dropping TB-1022/11308 due 

In [8]:
key='tb21_22_2984_pats_22_vars_result_at_end_of_treatment_wo_dr_reg'

'tb21_22_2984_pats_22_vars_result_at_end_of_treatment_wo_dr_reg'

In [8]:
fn=f'../data/{key}_{period_end_day}_days_{data_inclusion_type}_input_dict_all_ds_types_merged.json'
input_dict=json.load(open(fn))

In [12]:
input_dict['TB-1021/1084727']

'Microbiological test results of patient: [Day -1: LJ-slope: positive; LJ-slope culture status: positive; Zn-smear: 3+; MGIT: 7.958 Days, Positive; MGIT culture status: positive; Day 1: LJ-slope: 0.0 Cfu, Negative; LJ-slope culture status: positive; Zn-smear: 3+; MGIT: negative; MGIT culture status: positive; Day 8: LJ-slope: positive; LJ-slope culture status: positive; Zn-smear: 4+; MGIT: 38.333 Days, Positive; MGIT culture status: positive; Day 15: LJ-slope: positive; LJ-slope culture status: positive; Zn-smear: 4+; MGIT: 6.083 Days, Positive; MGIT culture status: positive; Day 22: LJ-slope: positive; LJ-slope culture status: positive; Zn-smear: 2+; MGIT: 23.458 Days, Positive; MGIT culture status: positive; Day 29: LJ-slope: 0.0 Cfu, Negative; LJ-slope culture status: positive; Zn-smear: 1+; MGIT: negative; MGIT culture status: positive; Day 36: LJ-slope: positive; LJ-slope culture status: positive; Zn-smear: 2+; MGIT: negative; MGIT culture status: positive; Day 43: LJ-slope: 10.0 